# Figure 2 and Supplementary Figures 4–6: cohort clock organization

This self-contained R notebook regenerates the figure analyses and all individual/assembled PNG and PDF exports from the current processed expression, phase, regression, and enrichment inputs. It embeds every plotting and statistical helper and does not source legacy notebooks or project scripts.

**Input boundary.** Starting inputs are merged log2 CPM (TMM), saved CHIRAL phases, current cohort rhythmic-gene tables, restricted-GTEx harmonic-regression products, and processed promoter-peak counts, gene-biotype/membership tables, and PPI tables. This notebook does not rerun alignment/counting, CHIRAL inference, ChIP peak-to-gene assignment, or raw BioGRID parsing. These upstream products must accompany a public release; their direct linkage to `process.smk` is not asserted. The input manifest records every required file and checksum.

**Execution.** Run all cells with the R (`ir`) kernel. Set `CHRONO_PROJECT_ROOT` and optionally `CHRONO_DATA_DIR` for a portable checkout. Outputs use `data/publication/fig02_cohort_clock_organization/` and `data/figures/publication/fig02_cohort_clock_organization/`.

**Publication conventions.** Global text roles are at least 7 pt, except the user-requested 5-pt B/D gene annotations. Layout uses 52.5 mm modules. PNGs omit panel letters; PDFs include them. Every individual panel and assembled figure is displayed inline. Source tables preserve identifiers, sample membership, statistics, and row order.

**Interpretation.** Samples are cross-sectional cohorts, ordered by inferred phase. Harmonic fits describe phase-associated expression and are not independent time-course validation. Axial R₂ denotes the magnitude of the second circular moment, not regression R-squared. ChIP enrichment colors and stars use BH-adjusted q-values (displayed as italic q); PPI enrichment stars use raw p-values.


## 1. Setup and processed-input quality control

In [ ]:
# Cell 1: Setup, paths, and publication style
required_packages <- c("dplyr", "tidyr", "tibble", "readr", "data.table", "ggplot2", "patchwork", "ragg",
    "ggrepel", "scales", "digest", "jsonlite", "stringr")
missing_packages <- required_packages[!vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_packages)) stop("Install required packages: ", paste(missing_packages, collapse = ", "))
suppressPackageStartupMessages(invisible(lapply(required_packages, library, character.only = TRUE)))
root_candidates <- unique(c(Sys.getenv("CHRONO_PROJECT_ROOT", ""), getwd(), dirname(getwd()), file.path(Sys.getenv("WORKSPACE_ROOT",
    "/home/wdeng3/workspace/Codespace"), "PDAC_Rhythmicity")))
root_candidates <- root_candidates[nzchar(root_candidates)]
root_candidates <- root_candidates[dir.exists(file.path(root_candidates, "notebook_publication"))]
if (!length(root_candidates)) stop("Set CHRONO_PROJECT_ROOT to the publication repository root.")
PROJECT_ROOT <- normalizePath(root_candidates[[1]], winslash = "/", mustWork = TRUE)
DATA_DIR <- Sys.getenv("CHRONO_DATA_DIR", file.path(PROJECT_ROOT, "data"))
ANALYSIS_SLUG <- "fig02_cohort_clock_organization"
TABLE_DIR <- file.path(PROJECT_ROOT, "data", "publication", ANALYSIS_SLUG)
FIGURE_DIR <- file.path(PROJECT_ROOT, "data", "figures", "publication", ANALYSIS_SLUG)
LOG_DIR <- file.path(PROJECT_ROOT, "data", "logs", "publication", ANALYSIS_SLUG)
invisible(lapply(c(TABLE_DIR, FIGURE_DIR, LOG_DIR), dir.create, recursive = TRUE, showWarnings = FALSE))
set.seed(213L)
RHYTHM_Q <- 0.05
RHYTHM_AMPLITUDE <- 0.1  # Peak-to-trough on log2 CPM (TMM).
COHORTS <- c("GTEx_Normal", "TCGA-PAAD_Tumor", "CPTAC-3_Tumor")
COHORT_COLOURS <- c(GTEx_Normal = "#456BAB", `TCGA-PAAD_Tumor` = "#F2B770", `CPTAC-3_Tumor` = "#C46456")
label_cohort <- function(x) unname(c(GTEx_Normal = "GTEx", `TCGA-PAAD_Tumor` = "TCGA", `CPTAC-3_Tumor` = "CPTAC")[as.character(x)])

# All physical sizes and text roles are configured here, in mm and points.
PANEL_MM <- 210/4
FONT_FAMILY <- "Helvetica"
FONT <- list(base = 8, title = 9, axis_title = 8, tick = 7, legend = 7, strip = 8, annotation = 7, tag = 10,
    subtitle = 7, gene_annotation = 5)
stopifnot(all(unlist(FONT[names(FONT) != "gene_annotation"]) >= 7), FONT$gene_annotation == 5)
publication_theme <- function() {
    theme_classic(base_size = FONT$base, base_family = FONT_FAMILY) + theme(text = element_text(size = FONT$base,
        family = FONT_FAMILY, colour = "black"), axis.title = element_text(size = FONT$axis_title), axis.text = element_text(size = FONT$tick,
        colour = "black"), axis.line = element_line(linewidth = 0.3), axis.ticks = element_line(linewidth = 0.3),
        legend.title = element_text(size = FONT$legend), legend.text = element_text(size = FONT$legend),
        strip.text = element_text(size = FONT$strip, face = "bold"), strip.background = element_rect(fill = "#F2F4F5",
            colour = NA), plot.title = element_text(size = FONT$title, face = "bold", hjust = 0.5), plot.subtitle = element_text(size = FONT$subtitle,
            hjust = 0.5), plot.title.position = "plot", plot.caption = element_text(size = FONT$annotation), plot.tag = element_text(size = FONT$tag,
            face = "bold"), plot.margin = margin(2, 2, 2, 2, unit = "mm"), legend.key = element_blank(),
        legend.background = element_blank())
}
theme_set(publication_theme())
EXPORT_MANIFEST <- list()
figure_font_sizes <- function(plot) {
    if (grDevices::dev.cur() == 1L) {
        grDevices::pdf(file = NULL)
        on.exit(grDevices::dev.off(), add = TRUE)
    }
    grob <- if (inherits(plot, "grob")) plot else if (inherits(plot, "patchwork"))
        patchwork::patchworkGrob(plot) else ggplot2::ggplotGrob(plot)
    collect <- function(x) {
        here <- if (inherits(x, "text"))
            x$gp$fontsize else if (inherits(x, "textrepeltree")) x$data$size * ggplot2::.pt else numeric()
        children <- c(if (!is.null(x$grobs)) x$grobs else list(), if (!is.null(x$children)) as.list(x$children) else list())
        c(here, unlist(lapply(children, collect), use.names = FALSE))
    }
    collect(grob)
}
save_publication <- function(plot, stem, width_units, height_units, pdf_plot = NULL, pdf_has_panel_labels = TRUE,
    display_inline = TRUE) {
    fractional_panel <- (stem == "figure2/Figure2K_clock_tf_chip_enrichment" && abs(height_units - 0.6) < 1e-9) ||
        (stem == "figure2/Figure2L_ccg_ppi_enrichment" && abs(height_units - 0.4) < 1e-9)
    stopifnot(width_units %in% 1:4, height_units > 0,
        fractional_panel || (height_units >= 1 && height_units == as.integer(height_units)))
    if (is.null(pdf_plot))
        stop("Provide the PDF plot for ", stem)
    font_sizes <- c(figure_font_sizes(plot), figure_font_sizes(pdf_plot))
    # Only the B/D gene annotations (and their assembly) have a user-authorized 5-pt exception.
    gene_label_exception <- stem %in% c("figure2/Figure2B_gtex_clock_amplitude_phase",
        "figure2/Figure2D_tcga_clock_amplitude_phase", "figure2/Figure2_assembled_cohort_clock_organization")
    minimum_allowed <- if (gene_label_exception) FONT$gene_annotation else 7
    if (!length(font_sizes) || any(font_sizes < minimum_allowed - 1e-06))
        stop("Text below the permitted size found in ", stem)
    width_mm <- PANEL_MM * width_units
    height_mm <- PANEL_MM * height_units
    out_stem <- file.path(FIGURE_DIR, stem)
    dir.create(dirname(out_stem), recursive = TRUE, showWarnings = FALSE)
    pdf_file <- paste0(out_stem, ".pdf")
    png_file <- paste0(out_stem, ".png")
    grDevices::pdf(pdf_file, width = width_mm/25.4, height = height_mm/25.4, family = FONT_FAMILY, bg = "white",
        onefile = TRUE, useDingbats = FALSE)
    tryCatch(if (inherits(pdf_plot, "grob")) grid::grid.draw(pdf_plot) else print(pdf_plot), finally = grDevices::dev.off())
    ragg::agg_png(png_file, width = width_mm, height = height_mm, units = "mm", res = 300, background = "white")
    tryCatch(print(plot), finally = grDevices::dev.off())
    EXPORT_MANIFEST[[length(EXPORT_MANIFEST) + 1L]] <<- tibble(stem = stem, width_modules = width_units,
        height_modules = height_units, width_mm = width_mm, height_mm = height_mm, png_has_panel_labels = FALSE,
        minimum_text_pt = min(font_sizes), pdf_has_panel_labels = pdf_has_panel_labels, png = file.path(stem |>
            dirname(), basename(png_file)), pdf = file.path(stem |>
            dirname(), basename(pdf_file)))
    if (display_inline && requireNamespace("IRdisplay", quietly = TRUE)) {
        IRdisplay::display_png(file = png_file, width = min(1100L, round(width_mm/25.4 * 110)))
    } else if (display_inline)
        print(plot)
    invisible(c(png = png_file, pdf = pdf_file))
}



# Shared Gaussian circular-distance KDE for main and restricted-GTEx phase plots.
legacy_density <- function(h, n = 200L, bw = 0.3) {
    th <- (h%%24) * 2 * pi/24
    g <- seq(0, 2 * pi, length.out = n)
    d <- vapply(g, function(v) {
        w <- abs(th - v)
        sum(exp(-pmin(w, 2 * pi - w)^2/(2 * bw^2)))
    }, numeric(1))
    data.frame(theta = g, density = d/(sum(d) * (2 * pi/n)))
}


In [ ]:
# Cell 2: Processed inputs and identifier QC
MERGED_DIR <- file.path(DATA_DIR, "Merged")
input_files <- c(metadata = file.path(MERGED_DIR, "GTEx_TCGA_GEO_merged_meta.csv"), phase = file.path(MERGED_DIR,
    "CHIRAL_dual_penalty_phi_12h_geo.csv"), expression = file.path(MERGED_DIR, "GTEx_TCGA_GEO_merged_TMM_unfiltered_expression.csv"),
    regression = file.path(MERGED_DIR, "regression_periodic_12h_geo.txt"))
stopifnot(all(file.exists(input_files)))
meta_raw <- as.data.frame(data.table::fread(input_files[["metadata"]], header = TRUE, check.names = FALSE))
phi_raw <- as.data.frame(data.table::fread(input_files[["phase"]], check.names = FALSE))
stopifnot(!anyDuplicated(meta_raw$Sample), !anyDuplicated(phi_raw$sample))
sample_meta <- tibble(sample_id = meta_raw$Sample, Group = meta_raw$Info) |>
    filter(Group %in% COHORTS) |>
    inner_join(tibble(sample_id = phi_raw$sample, phase_h = (phi_raw$phi%%(2 * pi)) * 12/pi), by = "sample_id") |>
    arrange(factor(Group, levels = COHORTS), phase_h, sample_id)
expr_header <- names(data.table::fread(input_files[["expression"]], nrows = 0, header = TRUE, check.names = FALSE))
missing_expression_samples <- setdiff(sample_meta$sample_id, expr_header)
if (length(missing_expression_samples)) stop("Expression missing cohort samples: ", paste(missing_expression_samples,
    collapse = ", "))
expr_df <- data.table::fread(input_files[["expression"]], select = c(expr_header[1], sample_meta$sample_id),
    header = TRUE, check.names = FALSE)
gene_ids <- as.character(expr_df[[1]])
stopifnot(!anyDuplicated(gene_ids), !anyDuplicated(sample_meta$sample_id), all(is.finite(sample_meta$phase_h)))
expr_matrix <- as.matrix(expr_df[, -1, with = FALSE])
rownames(expr_matrix) <- gene_ids
storage.mode(expr_matrix) <- "double"
rm(expr_df)
stopifnot(identical(colnames(expr_matrix), sample_meta$sample_id))
regression_all <- as_tibble(data.table::fread(input_files[["regression"]])) |>
    filter(Group %in% COHORTS)
sig_tables <- setNames(lapply(COHORTS, function(g) {
    f <- file.path(MERGED_DIR, "regression", paste0("sig_genes_regression_periodic_12h_", g, ".txt"))
    d <- as_tibble(data.table::fread(f))
    stopifnot(all(c("Gene", "ENSG", "acrophase", "Amplitude", "p", "q") %in% names(d)), !anyDuplicated(d$ENSG),
        all(is.finite(d$acrophase)), all(d$q < RHYTHM_Q), all(d$Amplitude > RHYTHM_AMPLITUDE))
    d |>
        mutate(acrophase = acrophase%%24)
}), COHORTS)
input_qc <- bind_rows(lapply(COHORTS, function(g) {
    samples <- sample_meta$sample_id[sample_meta$Group == g]
    d <- sig_tables[[g]]
    tibble(Group = g, samples = length(samples), phase_min_h = min(sample_meta$phase_h[sample_meta$Group ==
        g]), phase_max_h = max(sample_meta$phase_h[sample_meta$Group == g]), rhythmic_rows = nrow(d),
        unique_symbols = length(unique(d$Gene)), rhythmic_ENSG_in_expression = sum(d$ENSG %in% rownames(expr_matrix)),
        expression_nonfinite = sum(!is.finite(expr_matrix[, samples, drop = FALSE])))
}))
print(input_qc)
stopifnot(all(input_qc$samples > 0), all(input_qc$expression_nonfinite == 0))
write_csv(input_qc, file.path(TABLE_DIR, "input_cohort_qc.csv"))
write_csv(sample_meta, file.path(TABLE_DIR, "sample_phase_metadata.csv"))

missing_heatmap_genes <- bind_rows(lapply(COHORTS, function(g) sig_tables[[g]] |>
    filter(!ENSG %in% rownames(expr_matrix)) |>
    mutate(reason = "ENSG absent from current expression matrix")))
write_csv(missing_heatmap_genes, file.path(TABLE_DIR, "heatmap_unmatched_identifiers.csv"))



## 2. Current heatmaps and explicit cross-cohort gene order

Heatmaps retain current significant ENSG identifiers present in the expression matrix. Source QC records unmatched identifiers and empty phase bins. Within each cohort, expression is averaged in numeric 1 h phase bins and standardized across bins for each gene. Colors show row z-scores clipped to ±1; gray denotes an empty bin. Rows in Figure 2E and 2G are identical and ordered explicitly by GTEx acrophase (with ENSG as a stable tie-breaker); Figure 2G displays TCGA expression. Figure 2F uses TCGA rhythmic genes ordered by TCGA acrophase.


In [ ]:
# Cell 3: Current heatmaps and GTEx acrophase ordering
ordered_sig_genes <- function(group) {
    sig_tables[[group]] |>
        filter(ENSG %in% rownames(expr_matrix)) |>
        arrange(acrophase, ENSG)
}
gtex_order <- ordered_sig_genes("GTEx_Normal")
tcga_order <- ordered_sig_genes("TCGA-PAAD_Tumor")
make_heatmap_source <- function(expression_group, ordered_genes, panel) {
    sm <- sample_meta |>
        filter(Group == expression_group) |>
        mutate(hour_bin = floor(phase_h)%%24)
    x <- expr_matrix[ordered_genes$ENSG, sm$sample_id, drop = FALSE]
    binned <- matrix(NA_real_, nrow = nrow(x), ncol = 24, dimnames = list(rownames(x), as.character(0:23)))
    for (h in 0:23) {
        ii <- which(sm$hour_bin == h)
        if (length(ii))
            binned[, h + 1L] <- rowMeans(x[, ii, drop = FALSE])
    }
    centered <- binned - rowMeans(binned, na.rm = TRUE)
    row_sd <- sqrt(rowSums(centered^2, na.rm = TRUE)/(rowSums(is.finite(centered)) - 1))
    z <- centered/ifelse(is.finite(row_sd) & row_sd > 0, row_sd, 1)
    order_table <- ordered_genes |>
        transmute(row_index = row_number(), ENSG, Gene, ordering_acrophase_h = acrophase, ordering_cohort = if (panel ==
            "F")
            "TCGA-PAAD_Tumor" else "GTEx_Normal", expression_cohort = expression_group)
    bins <- tibble(hour_bin = 0:23, n_samples = tabulate(sm$hour_bin + 1L, nbins = 24), Group = expression_group,
        panel = panel)
    write_csv(order_table, file.path(TABLE_DIR, paste0("Figure2", panel, "_heatmap_gene_order.csv")))
    write_csv(bind_cols(tibble(ENSG = rownames(z)), as_tibble(z, .name_repair = "minimal")), file.path(TABLE_DIR,
        paste0("Figure2", panel, "_heatmap_row_z_scores.csv")))
    write_csv(bins, file.path(TABLE_DIR, paste0("Figure2", panel, "_heatmap_bin_counts.csv")))
    list(z = z, order = order_table, bins = bins)
}
heat_e <- make_heatmap_source("GTEx_Normal", gtex_order, "E")
heat_f <- make_heatmap_source("TCGA-PAAD_Tumor", tcga_order, "F")
heat_g <- make_heatmap_source("TCGA-PAAD_Tumor", gtex_order, "G")
stopifnot(identical(heat_e$order$ENSG, heat_g$order$ENSG), all(diff(heat_g$order$ordering_acrophase_h) >=
    0))
heatmap_qc <- tibble(panel = c("E", "F", "G"), genes = c(nrow(heat_e$z), nrow(heat_f$z), nrow(heat_g$z)),
    expression_cohort = c("GTEx_Normal", "TCGA-PAAD_Tumor", "TCGA-PAAD_Tumor"), ordering_cohort = c("GTEx_Normal",
        "TCGA-PAAD_Tumor", "GTEx_Normal"))
print(heatmap_qc)
write_csv(heatmap_qc, file.path(TABLE_DIR, "heatmap_qc.csv"))
make_publication_heatmap <- function(src, title, subtitle) {
    color_fun <- scales::col_numeric(c("#2166AC", "white", "#B2182B"), domain = c(-1, 1), na.color = "#D9D9D9")
    colors <- matrix(color_fun(pmax(-1, pmin(1, src$z))), nrow = nrow(src$z), ncol = 24)
    # A single raster retains every source row without allocating hundreds of thousands of vector
    # tiles.
    ggplot() + annotation_raster(as.raster(colors), xmin = 0, xmax = 24, ymin = 0, ymax = 1, interpolate = FALSE) +
        geom_point(data = tibble(x = c(0, 24), y = 0, z = c(-1, 1)), aes(x, y, fill = z), shape = 22,
            alpha = 0, show.legend = TRUE) + scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#B2182B",
        limits = c(-1, 1), breaks = c(-1, 0, 1), name = "Row z-score", oob = scales::squish, na.value = "#D9D9D9") +
        scale_x_continuous(limits = c(0, 24), breaks = c(0, 6, 12, 18, 24), expand = c(0, 0)) + scale_y_continuous(limits = c(0,
        1), breaks = NULL, expand = c(0, 0)) + labs(title = title, subtitle = paste0(subtitle, "; n = ",
        format(nrow(src$z), big.mark = ",")), x = "Phase (h)", y = NULL) + guides(fill = guide_colourbar(direction = "horizontal",
        title.position = "top", barwidth = grid::unit(22, "mm"), barheight = grid::unit(2, "mm"))) +
        publication_theme() + theme(axis.line.y = element_blank(), axis.ticks.y = element_blank(), legend.position = "bottom",
        legend.margin = margin(0, 0, 0, 0), legend.box.spacing = grid::unit(0, "mm"), plot.title = element_text(size = FONT$title),
        plot.subtitle = element_text(size = FONT$subtitle), plot.margin = margin(2, 1.5, 1, 1.5, unit = "mm"))
}
fig2_e <- make_publication_heatmap(heat_e, "GTEx", "GTEx acrophase order")
fig2_f <- make_publication_heatmap(heat_f, "TCGA", "TCGA acrophase order")
fig2_g <- make_publication_heatmap(heat_g, "TCGA expression of\nGTEx rhythmic genes", "GTEx acrophase order")



### Figure 2 and Supplementary Figures 4–5: clock-gene phase organization

Clock-gene panels use prespecified genes without filtering them for rhythmic significance. Figure 2A/C and S4 re-fit 24 h harmonic OLS models to merged log2 CPM (TMM) expression against the saved CHIRAL phases. Figure 2B/D and S5D show all 13 mapped clock genes. S5A–C retain the historical 12-gene SVD set containing CIART and omitting CLOCK; S5H retains the restricted-GTEx 12-gene set containing CLOCK and omitting CIART. A single-cohort complex-coefficient SVD has one column and therefore has 100% mode-1 variance by construction. These prespecified sets remain distinct from the cohort-wide rhythmic sets used below.


In [ ]:
# Cell 4: Core-clock expression and harmonic fits
suppressPackageStartupMessages({
    library(ggplot2)
    library(dplyr)
    library(tidyr)
    library(ggrepel)
    library(readr)
})

# Explicit, version-free identifiers for every displayed gene.
clock_scatter <- c(BMAL1 = "ENSG00000133794", CLOCK = "ENSG00000134852", NPAS2 = "ENSG00000170485", CRY1 = "ENSG00000008405",
    CRY2 = "ENSG00000121671", PER1 = "ENSG00000179094", PER2 = "ENSG00000132326", PER3 = "ENSG00000049246",
    NR1D1 = "ENSG00000126368", NR1D2 = "ENSG00000174738", DBP = "ENSG00000105516", TEF = "ENSG00000167074")
clock_polar <- c(clock_scatter, CIART = "ENSG00000159208")
clock_svd_main <- c(BMAL1 = "ENSG00000133794", NPAS2 = "ENSG00000170485", CRY1 = "ENSG00000008405", CRY2 = "ENSG00000121671",
    PER1 = "ENSG00000179094", PER2 = "ENSG00000132326", PER3 = "ENSG00000049246", NR1D1 = "ENSG00000126368",
    NR1D2 = "ENSG00000174738", DBP = "ENSG00000105516", TEF = "ENSG00000167074", CIART = "ENSG00000159208")
clock_svd_sensitivity <- clock_scatter
# Verify hard-coded symbols against the project annotation before any extraction.
annotation_qc <- readr::read_tsv(file.path(DATA_DIR, "Ref", "genes.bed"), col_names = c("chr", "start",
    "end", "gene", "score", "strand"), show_col_types = FALSE) %>%
    tidyr::separate(gene, into = c("ENSG", "annotation_symbol"), sep = ":", remove = TRUE)
clock_id_qc <- tibble(symbol = names(clock_polar), ENSG = unname(clock_polar)) %>%
    left_join(annotation_qc %>%
        select(ENSG, annotation_symbol), by = "ENSG") %>%
    mutate(expected_symbol = if_else(symbol == "BMAL1", "BMAL1", symbol))
stopifnot(nrow(clock_id_qc) == length(clock_polar), all(clock_id_qc$annotation_symbol == clock_id_qc$expected_symbol))
write_tsv(clock_id_qc, file.path(TABLE_DIR, "figure2_clock_gene_identifier_qc.tsv"))
cohort_order <- c("GTEx_Normal", "TCGA-PAAD_Tumor", "CPTAC-3_Tumor")
cohort_palette <- unname(COHORT_COLOURS[cohort_order])
names(cohort_palette) <- cohort_order

expr_matrix <- as.matrix(expr_matrix)
stopifnot(is.numeric(expr_matrix), all(c("sample_id", "Group", "phase_h") %in% names(sample_meta)))
sample_meta <- sample_meta %>%
    filter(Group %in% cohort_order, is.finite(phase_h)) %>%
    mutate(phase_h = phase_h%%24)
shared_samples <- intersect(colnames(expr_matrix), sample_meta$sample_id)
stopifnot(length(shared_samples) > 4L)
sample_meta <- sample_meta %>%
    filter(sample_id %in% shared_samples) %>%
    arrange(match(sample_id, shared_samples))

harmonic_fit <- function(y, phase_h) {
    keep <- is.finite(y) & is.finite(phase_h)
    y <- as.numeric(y[keep])
    phase_h <- as.numeric(phase_h[keep])%%24
    if (length(y) < 5L)
        return(NULL)
    fit <- lm(y ~ cos(2 * pi * phase_h/24) + sin(2 * pi * phase_h/24))
    b <- coef(fit)
    fs <- summary(fit)$fstatistic
    p <- pf(fs[1], fs[2], fs[3], lower.tail = FALSE)
    list(n = length(y), intercept = b[1], cos_coef = b[2], sin_coef = b[3], amplitude = sqrt(b[2]^2 +
        b[3]^2), p = p, acrophase = (atan2(b[3], b[2]) * 12/pi)%%24)
}
fit_clock_cohort <- function(ids, group) {
    md <- sample_meta %>%
        filter(Group == group)
    out <- lapply(names(ids), function(symbol) {
        id <- ids[[symbol]]
        if (!id %in% rownames(expr_matrix))
            return(NULL)
        z <- harmonic_fit(expr_matrix[id, md$sample_id], md$phase_h)
        if (is.null(z))
            return(NULL)
        tibble(symbol = symbol, ENSG = id, Group = group, n = z$n, intercept = z$intercept, cos_coef = z$cos_coef,
            sin_coef = z$sin_coef, Amplitude = z$amplitude, p = z$p, acrophase = z$acrophase)
    })
    bind_rows(out)
}
clock_fit_all <- bind_rows(lapply(cohort_order, function(g) fit_clock_cohort(clock_polar, g)))
write_tsv(clock_fit_all, file.path(TABLE_DIR, "figure2_core_clock_harmonic_fit_statistics.tsv"))

phase_curve <- function(stats, n = 241L) {
    grid <- seq(0, 24, length.out = n)
    stats %>%
        rowwise() %>%
        do(tibble(symbol = .$symbol, Group = .$Group, phase_h = grid, fitted = .$intercept + .$cos_coef *
            cos(2 * pi * grid/24) + .$sin_coef * sin(2 * pi * grid/24))) %>%
        ungroup()
}
expression_fit_annotations <- function(ann, italic_p = TRUE) {
    if (!italic_p) return(list(geom_text(data = ann, aes(x, y, label = label), inherit.aes = FALSE,
        hjust = 0, vjust = 1, size = FONT$annotation/ggplot2::.pt)))
    ann$label_main <- sprintf("n = %d\nAmp. = %.2f", ann$n, ann$Amplitude)
    ann$label_p <- sprintf("italic(p) == '%s'", ifelse(ann$p < 0.001, sprintf("%.1e", ann$p), sprintf("%.3f", ann$p)))
    list(geom_text(data = ann, aes(x, y, label = label_main), inherit.aes = FALSE,
            hjust = 0, vjust = 1, size = FONT$annotation/ggplot2::.pt),
        geom_text(data = ann, aes(x, y, label = label_p), inherit.aes = FALSE,
            hjust = 0, vjust = 4.6, size = FONT$annotation/ggplot2::.pt, parse = TRUE))
}

make_expression_panel <- function(group, genes, ncol, italic_p = TRUE) {
    md <- sample_meta %>%
        filter(Group == group)
    long <- tibble(sample_id = md$sample_id, phase_h = md$phase_h) %>%
        crossing(symbol = names(genes)) %>%
        mutate(symbol = factor(symbol, levels = names(genes)), ENSG = unname(genes[as.character(symbol)]),
            expr = mapply(function(id, s) expr_matrix[id, s], ENSG, sample_id)) %>%
        filter(is.finite(expr))
    stats <- clock_fit_all %>%
        filter(Group == group, symbol %in% names(genes))
    curves <- phase_curve(stats)
    ybounds <- long %>%
        group_by(symbol) %>%
        summarise(y = max(expr) + 0.12 * diff(range(expr)), .groups = "drop")
    ann <- stats %>%
        left_join(ybounds, by = "symbol") %>%
        mutate(label = sprintf("n = %d\nAmp. = %.2f\np = %s", n, Amplitude, ifelse(p < 0.001, sprintf("%.1e",
            p), sprintf("%.3f", p))), x = 0.35)
    ggplot(long, aes(phase_h, expr)) + geom_point(colour = cohort_palette[[group]], alpha = 0.5, size = 0.65) +
        geom_line(data = curves, aes(phase_h, fitted), colour = cohort_palette[[group]], linewidth = 0.5,
            inherit.aes = FALSE) + expression_fit_annotations(ann, italic_p) + facet_wrap(~symbol, ncol = ncol, scales = "free_y") +
        scale_x_continuous(breaks = seq(0, 24, 6), limits = c(0, 24), expand = c(0, 0)) + scale_y_continuous(expand = expansion(mult = c(0.04,
        0.16))) + labs(title = label_cohort(group), subtitle = sprintf("n = %d; 24 h harmonic OLS fit", nrow(md)), x = "CHIRAL phase (h)",
        y = "log2 CPM (TMM)") + publication_theme() + theme(plot.title = element_text(size = FONT[["title"]],
        face = "bold"), plot.subtitle = element_text(size = FONT[["subtitle"]]), strip.text = element_text(size = FONT[["strip"]],
        face = "bold"), axis.title = element_text(size = FONT[["axis_title"]]), axis.text = element_text(size = FONT[["tick"]]))
}
fig2_a <- make_expression_panel("GTEx_Normal", clock_scatter[c("BMAL1", "CRY2", "PER3")], 3, italic_p = TRUE)
fig2_c <- make_expression_panel("TCGA-PAAD_Tumor", clock_scatter[c("BMAL1", "CRY2", "PER3")], 3, italic_p = TRUE)
s4_a <- make_expression_panel("TCGA-PAAD_Tumor", clock_scatter, 4)
s4_b <- make_expression_panel("GTEx_Normal", clock_scatter, 4)



In [ ]:
# Cell 5: Core-clock amplitudes and SVD
polar_metrics <- clock_fit_all %>%
    transmute(Group, symbol, Gene = symbol, ENSG, acrophase, Amplitude = 2 * Amplitude, p, q = NA_real_)
stopifnot(nrow(polar_metrics) == length(cohort_order) * length(clock_polar))
write_tsv(polar_metrics, file.path(TABLE_DIR, "figure2_s5_core_clock_polar_source.tsv"))
make_polar_panel <- function(group) {
    d <- polar_metrics %>%
        filter(Group == group) %>%
        mutate(neglog10p = -log10(pmax(p, .Machine$double.xmin)))
    ggplot(d, aes(acrophase, Amplitude, fill = neglog10p, label = symbol)) + geom_point(shape = 21, colour = "black",
        stroke = 0.25, size = 2.1) + ggrepel::geom_text_repel(size = FONT[["annotation"]]/.pt, seed = 20260921,
        max.overlaps = Inf, min.segment.length = 0) + coord_polar(theta = "x", start = 0) + scale_x_continuous(breaks = seq(0,
        24, 4), labels = seq(0, 24, 4), limits = c(0, 24), expand = c(0, 0)) + scale_fill_viridis_c(option = "magma",
        name = expression(-log[10](italic(p)))) + labs(title = label_cohort(group), x = "Acrophase (h)", y = "Amplitude") +
        publication_theme() + theme(plot.title = element_text(size = FONT[["title"]], face = "bold"),
        axis.title = element_text(size = FONT[["axis_title"]]), axis.text = element_text(size = FONT[["tick"]]),
        legend.title = element_text(size = FONT[["legend"]]), legend.text = element_text(size = FONT[["legend"]]))
}
fig2_b <- make_polar_panel("GTEx_Normal")
fig2_d <- make_polar_panel("TCGA-PAAD_Tumor")
# Main-cohort SVD panels use one complex coefficient vector per cohort.
make_svd_panel <- function(group) {
    d <- clock_fit_all %>%
        filter(Group == group, ENSG %in% unname(clock_svd_main)) %>%
        mutate(symbol = factor(symbol, levels = names(clock_svd_main))) %>%
        arrange(symbol)
    stopifnot(nrow(d) == length(clock_svd_main), all(is.finite(d$cos_coef)), all(is.finite(d$sin_coef)))
    z <- svd(matrix(complex(real = d$cos_coef, imaginary = d$sin_coef), ncol = 1L))
    rownames(z$u) <- as.character(d$symbol)
    loadings <- tibble(Group = group, symbol = rownames(z$u), phase_h = (Arg(z$u[, 1]) * 12/pi)%%24,
        loading = Mod(z$u[, 1]), mode = 1L, variance_explained = z$d[1]^2/sum(z$d^2))
    write_tsv(loadings, file.path(TABLE_DIR, paste0("figure_s5_", group, "_core_clock_svd_mode1.tsv")))
    ggplot(loadings, aes(phase_h, loading, label = symbol)) + geom_point(colour = cohort_palette[[group]],
        size = 2) + ggrepel::geom_text_repel(size = FONT[["annotation"]]/.pt, seed = 20260921, max.overlaps = Inf,
        min.segment.length = 0) + coord_polar(theta = "x", start = 0) + scale_x_continuous(breaks = seq(0,
        24, 4), limits = c(0, 24), expand = c(0, 0)) + labs(title = label_cohort(group), subtitle = "Mode 1 (100%)",
        x = "Component phase (h)", y = "Loading modulus") + publication_theme() + theme(plot.title = element_text(size = FONT[["title"]],
        face = "bold"), plot.subtitle = element_text(size = FONT[["subtitle"]]), axis.title = element_text(size = FONT[["axis_title"]]),
        axis.text = element_text(size = FONT[["tick"]]))
}
s5_a <- make_svd_panel("GTEx_Normal")
s5_b <- make_svd_panel("TCGA-PAAD_Tumor")
s5_c <- make_svd_panel("CPTAC-3_Tumor")
s5_d <- make_polar_panel("CPTAC-3_Tumor")



### Restricted GTEx sensitivity

This sensitivity analysis reads only saved CHIRAL phase and harmonic-regression products from the ToD [0, 12) donor selection. It does not rerun phase inference. Rhythmic genes are reselected from the current restricted-cohort regression table using the same BH q < 0.05 and peak-to-trough amplitude > 0.1 rule as the main cohorts. Density uses the same Gaussian circular-distance kernel (bandwidth 0.3 radians) as Figure 2I and the paired dashed radii mark the axial second-moment orientation.

In [ ]:
# Cell 6: Restricted-GTEx sensitivity
sensitivity_dir <- file.path(DATA_DIR, "gtex_chiral_sensitivity_supplement")
sens_regression <- readr::read_tsv(file.path(sensitivity_dir, "harmonic_regression_all_genes.tsv"), show_col_types = FALSE)
sens_rhythmic <- readr::read_tsv(file.path(sensitivity_dir, "rhythmic_genes.tsv"), show_col_types = FALSE)
sens_phase <- readr::read_tsv(file.path(sensitivity_dir, "chiral_phase_estimates.tsv"), show_col_types = FALSE)
stopifnot(all(c("gene_id", "cos_coef", "sin_coef", "GENENAME") %in% names(sens_regression)), "acrophase_hours" %in%
    names(sens_rhythmic), "phase_hours" %in% names(sens_phase))
sens_current <- sens_regression |> filter(is.finite(q), is.finite(amplitude),
    q < RHYTHM_Q, amplitude > RHYTHM_AMPLITUDE)
stopifnot(setequal(sens_current$gene_id, sens_rhythmic$gene_id))
sens_rhythmic <- sens_current
write_tsv(sens_rhythmic, file.path(TABLE_DIR, "s5_sensitivity_current_rhythmic_genes.tsv"))
write_tsv(sens_phase, file.path(TABLE_DIR, "s5_sensitivity_chiral_phase_source.tsv"))

sens_clock <- tibble(symbol = names(clock_svd_sensitivity), gene_id = unname(clock_svd_sensitivity)) %>%
    left_join(sens_regression, by = "gene_id") %>%
    arrange(match(symbol, names(clock_svd_sensitivity)))
stopifnot(nrow(sens_clock) == length(clock_svd_sensitivity), all(is.finite(sens_clock$cos_coef)), all(is.finite(sens_clock$sin_coef)))
sens_svd <- svd(matrix(complex(real = sens_clock$cos_coef, imaginary = sens_clock$sin_coef), ncol = 1L))
rownames(sens_svd$u) <- sens_clock$symbol
sens_svd_table <- tibble(symbol = rownames(sens_svd$u), phase_h = (Arg(sens_svd$u[, 1]) * 12/pi)%%24,
    loading = Mod(sens_svd$u[, 1]), mode = 1L, variance_explained = sens_svd$d[1]^2/sum(sens_svd$d^2))
write_tsv(sens_svd_table, file.path(TABLE_DIR, "s5_sensitivity_core_clock_svd_mode1.tsv"))
s5_h <- ggplot(sens_svd_table, aes(phase_h, loading, label = symbol)) + geom_point(colour = COHORT_COLOURS[["GTEx_Normal"]],
    size = 2.1) + ggrepel::geom_text_repel(size = FONT[["annotation"]]/.pt, seed = 20260921, max.overlaps = Inf,
    min.segment.length = 0) + coord_polar(theta = "x", start = 0) + scale_x_continuous(breaks = seq(0,
    24, 4), limits = c(0, 24), expand = c(0, 0)) + labs(title = "GTEx ToD [0, 12): core-clock SVD", subtitle = "Mode 1 (100%)",
    x = "Component phase (h)", y = "Loading modulus") + publication_theme() + theme(plot.title = element_text(size = FONT[["title"]],
    face = "bold"), plot.subtitle = element_text(size = FONT[["subtitle"]]), axis.title = element_text(size = FONT[["axis_title"]]),
    axis.text = element_text(size = FONT[["tick"]]))

sens_phase_values <- sens_rhythmic$acrophase_hours%%24
sens_density <- legacy_density(sens_phase_values, bw = 0.3)
z2_re <- mean(cos(2 * sens_phase_values * 2 * pi/24))
z2_im <- mean(sin(2 * sens_phase_values * 2 * pi/24))
axis_rad <- (atan2(z2_im, z2_re)/2)%%pi
sens_density_stats <- tibble(n_rhythmic = length(sens_phase_values), axial_R2 = sqrt(z2_re^2 + z2_im^2),
    axis_hours = axis_rad * 12/pi, opposite_axis_hours = (axis_rad * 12/pi + 12)%%24, kernel = "Gaussian circular-distance (same as Figure 2I)",
    bandwidth = 0.3)
write_tsv(sens_density, file.path(TABLE_DIR, "s5_sensitivity_rhythmic_gene_phase_density.tsv"))
write_tsv(sens_density_stats, file.path(TABLE_DIR, "s5_sensitivity_rhythmic_gene_phase_density_statistics.tsv"))
s5_i <- ggplot(sens_density, aes(theta, density)) + geom_area(fill = COHORT_COLOURS[["GTEx_Normal"]], alpha = 0.22) + geom_line(colour = COHORT_COLOURS[["GTEx_Normal"]],
    linewidth = 0.6) + geom_vline(xintercept = c(axis_rad, axis_rad + pi), colour = COHORT_COLOURS[["GTEx_Normal"]], linetype = "dashed",
    linewidth = 0.45) + coord_polar(theta = "x", start = 0) + scale_x_continuous(breaks = seq(0, 2 *
    pi, by = pi/3), labels = c("0", "4", "8", "12", "16", "20", "24"), expand = c(0, 0)) + labs(title = "GTEx ToD [0, 12): rhythmic-gene phases",
    subtitle = bquote("n = " * .(sens_density_stats$n_rhythmic) * "; axial " * R[textstyle(2)] * " = " * .(round(sens_density_stats$axial_R2, 2))),
    x = "Acrophase (h)", y = "Density") + publication_theme() + theme(plot.title = element_text(size = FONT[["title"]],
    face = "bold"), plot.subtitle = element_text(size = FONT[["subtitle"]]), axis.title = element_text(size = FONT[["axis_title"]]),
    axis.text = element_text(size = FONT[["tick"]]))



### Overlap, polarization, and common-versus-unique rhythmic genes

The shared rhythmicity rule is cohort-wise BH q < 0.05 and peak-to-trough amplitude > 0.1 (log2 CPM), configured globally in Cell 1. Figure 2H–J, S5E–G and S6 use all current rhythmic genes, deduplicated by current gene symbol (`Gene`). Figure 2E/F additionally require expression-matrix membership; Figure 2G uses GTEx membership and order while displaying TCGA expression. S5I applies the same rule within the separate restricted-GTEx regression analysis. S5J–K use current tumor-specific subsets defined in the enrichment section. All statistics and source/QC tables are regenerated.


In [ ]:
# Cell 7: Circular statistics and overlap helpers
suppressPackageStartupMessages(library(ggplot2))
# Requires shared PROJECT_ROOT, DATA_DIR, TABLE_DIR, FONT, FONT_FAMILY, publication_theme(),
# COHORT_COLOURS, label_cohort(), sig_tables.
stopifnot(is.list(sig_tables), all(c("GTEx_Normal", "TCGA-PAAD_Tumor", "CPTAC-3_Tumor") %in% names(sig_tables)))
dir.create(TABLE_DIR, recursive = TRUE, showWarnings = FALSE)
clean_sig <- function(x, nm) {
    z <- c("Gene", "ENSG", "acrophase", "Amplitude", "p")
    if (length(q <- setdiff(z, names(x))))
        stop(nm, " missing: ", paste(q, collapse = ", "))
    x <- as.data.frame(x)[, z]
    x$Gene <- trimws(as.character(x$Gene))
    x <- x[nzchar(x$Gene) & is.finite(x$acrophase) & is.finite(x$Amplitude) & is.finite(x$p), ]
    x$acrophase <- x$acrophase%%24
    x$cohort <- nm
    x[!duplicated(x$Gene), ]
}
sig <- setNames(lapply(names(sig_tables), function(nm) clean_sig(sig_tables[[nm]], nm)), names(sig_tables))
gtex <- sig[["GTEx_Normal"]]
tcga <- sig[["TCGA-PAAD_Tumor"]]
cptac <- sig[["CPTAC-3_Tumor"]]
sets <- lapply(sig[c("GTEx_Normal", "TCGA-PAAD_Tumor", "CPTAC-3_Tumor")], function(x) x$Gene)
write.csv(data.frame(cohort = names(sets), n_genes = lengths(sets)), file.path(TABLE_DIR, "rhythmic_gene_count_qc.csv"),
    row.names = FALSE)
theta <- function(h) (h%%24) * 2 * pi/24
r2sum <- function(h) {
    z <- c(mean(cos(2 * theta(h))), mean(sin(2 * theta(h))))
    a <- (atan2(z[2], z[1])/2)%%pi
    data.frame(n_genes = length(h), R2 = sqrt(sum(z^2)), axis_hours = a * 24/(2 * pi), opposite_axis_hours = (a *
        24/(2 * pi) + 12)%%24)
}
boot_r2 <- function(h, B = 5000L, seed = 1L) {
    set.seed(seed)
    th <- theta(h)
    x <- cos(2 * th)
    y <- sin(2 * th)
    n <- length(h)
    vapply(seq_len(B), function(i) {
        j <- sample.int(n, n, TRUE)
        sqrt(mean(x[j])^2 + mean(y[j])^2)
    }, numeric(1))
}
# Precompute harmonic components: permutations only resample membership, avoiding repeated complex
# exponentials.
perm_delta <- function(a, b, P = 20000L, seed = 2001L) {
    x <- c(cos(2 * theta(a)), cos(2 * theta(b)))
    y <- c(sin(2 * theta(a)), sin(2 * theta(b)))
    na <- length(a)
    n <- length(x)
    sx <- sum(x)
    sy <- sum(y)
    f <- function(xx, yy, nn) sqrt((xx/nn)^2 + (yy/nn)^2)
    o <- f(sum(x[seq_len(na)]), sum(y[seq_len(na)]), na) - f(sum(x[(na + 1L):n]), sum(y[(na + 1L):n]),
        n - na)
    set.seed(seed)
    nul <- vapply(seq_len(P), function(i) {
        j <- sample.int(n, na)
        xx <- sum(x[j])
        yy <- sum(y[j])
        f(xx, yy, na) - f(sx - xx, sy - yy, n - na)
    }, numeric(1))
    list(delta = o, p = (sum(abs(nul) >= abs(o)) + 1)/(P + 1), null = nul)
}
cohens_d <- function(x, y) (mean(x) - mean(y))/sqrt(((length(x) - 1) * var(x) + (length(y) - 1) * var(y))/(length(x) +
    length(y) - 2))
w1 <- function(th, p, q) {
    if (tail(th, 1) == 2 * pi) {
        th <- th[-length(th)]
        p <- p[-length(p)]
        q <- q[-length(q)]
    }
    m <- length(th)
    d <- 2 * pi/m
    p <- pmax(p, 0) * d
    q <- pmax(q, 0) * d
    p <- p/sum(p)
    q <- q/sum(q)
    min(vapply(0:(m - 1L), function(s) {
        pp <- if (s == 0) p else c(p[(s + 1):m], p[seq_len(s)])
        qq <- if (s == 0) q else c(q[(s + 1):m], q[seq_len(s)])
        sum(abs(cumsum(pp) - cumsum(qq))) * d
    }, numeric(1)))
}
circle <- function(x, y, r, g) data.frame(x = x + r * cos(seq(0, 2 * pi, length.out = 181)), y = y +
    r * sin(seq(0, 2 * pi, length.out = 181)), group = g)



#### Figure 2H–J: GTEx/TCGA overlap and acrophase polarization


In [ ]:
# Cell 8: GTEx–TCGA overlap and polarization
ov2 <- function(a, b, left, right) data.frame(category = c(paste(left, "only"), "Shared", paste(right,
    "only")), n_genes = c(length(setdiff(a, b)), length(intersect(a, b)), length(setdiff(b, a))))
venn2 <- function(z, left, right, lc, rc) {
    cc <- rbind(circle(-0.62, 0, 1, "left"), circle(0.62, 0, 1, "right"))
    ggplot() + geom_polygon(data = cc, aes(x, y, group = group, fill = group), alpha = 0.28, colour = NA) +
        geom_path(data = cc, aes(x, y, group = group, colour = group), linewidth = 0.45) + geom_text(data = data.frame(x = c(-1.05,
        0, 1.05), y = 0, label = z$n_genes), aes(x, y, label = label), family = FONT_FAMILY, size = FONT$annotation/.pt,
        fontface = "bold") + annotate("text", x = c(-0.75, 0.75), y = 1.18, label = c(left, right), colour = c(lc,
        rc), family = FONT_FAMILY, size = FONT$legend/.pt) + scale_fill_manual(values = c(left = lc,
        right = rc), guide = "none") + scale_colour_manual(values = c(left = lc, right = rc), guide = "none") +
        coord_fixed(xlim = c(-1.75, 1.75), ylim = c(-1.3, 1.4), clip = "off") + publication_theme() +
        theme(axis.text = element_blank(), axis.title = element_blank(), panel.grid = element_blank())
}
overlap_bar <- function(z, reference, tumor) {
    d <- data.frame(cohort = c("Reference", "Tumor", "Tumor"), part = c("Reference", "Shared", "Unique"),
        n_genes = c(sum(z$n_genes[1:2]), z$n_genes[2], z$n_genes[3]))
    ggplot(d, aes(cohort, n_genes, fill = part)) + geom_col(width = 0.62) + geom_text(aes(label = n_genes),
        position = position_stack(vjust = 0.5), family = FONT_FAMILY, size = FONT$annotation/.pt) + scale_fill_manual(values = c(Reference = COHORT_COLOURS[[reference]],
        Shared = COHORT_COLOURS[[tumor]], Unique = COHORT_COLOURS[[tumor]]), labels = c(Reference = "GTEx rhythmic",
        Shared = "Shared", Unique = "Tumor only")) + scale_x_discrete(labels = c(Reference = label_cohort(reference),
        Tumor = label_cohort(tumor))) + labs(x = NULL, y = "Rhythmic genes", fill = NULL) + publication_theme() +
        theme(legend.position = "bottom")
}
overlapGTExTCGA <- ov2(sets$GTEx_Normal, sets[["TCGA-PAAD_Tumor"]], "GTEx", "TCGA")
write.csv(overlapGTExTCGA, file.path(TABLE_DIR, "fig2_h_gtex_tcga_overlap.csv"), row.names = FALSE)
fig2_h <- overlap_bar(overlapGTExTCGA, "GTEx_Normal", "TCGA-PAAD_Tumor")
dens2 <- rbind(transform(legacy_density(gtex$acrophase), cohort = "GTEx_Normal"), transform(legacy_density(tcga$acrophase),
    cohort = "TCGA-PAAD_Tumor"))
polar_qc <- rbind(cbind(cohort = "GTEx_Normal", r2sum(gtex$acrophase)), cbind(cohort = "TCGA-PAAD_Tumor",
    r2sum(tcga$acrophase)))
write.csv(dens2, file.path(TABLE_DIR, "fig2_i_acrophase_density.csv"), row.names = FALSE)
fig2_i <- ggplot(dens2, aes(theta, density, colour = cohort)) + geom_line(linewidth = 0.45) + coord_polar(theta = "x") +
    scale_x_continuous(breaks = seq(0, 2 * pi, pi/3), labels = c("0", "4", "8", "12", "16", "20", "24"),
        expand = c(0, 0)) + scale_colour_manual(values = c(GTEx_Normal = COHORT_COLOURS[["GTEx_Normal"]],
    `TCGA-PAAD_Tumor` = COHORT_COLOURS[["TCGA-PAAD_Tumor"]]), labels = label_cohort) + labs(x = NULL,
    y = "Density", colour = NULL) + publication_theme() + theme(legend.position = "bottom")
b <- list(boot_r2(gtex$acrophase, 5000L, 1001L), boot_r2(tcga$acrophase, 5000L, 1002L))
polar_qc$R2_lo <- vapply(b, quantile, numeric(1), probs = 0.025)
polar_qc$R2_hi <- vapply(b, quantile, numeric(1), probs = 0.975)
write.csv(polar_qc, file.path(TABLE_DIR, "fig2_j_axial_bootstrap_qc.csv"), row.names = FALSE)
fig2_j <- ggplot(polar_qc, aes(factor(cohort, levels = c("TCGA-PAAD_Tumor", "GTEx_Normal")), R2, fill = cohort)) + geom_col(width = 0.58) + geom_errorbar(aes(ymin = R2_lo,
    ymax = R2_hi), width = 0.13, linewidth = 0.35) + geom_text(aes(label = sprintf("%.2f", R2)), vjust = -0.55,
    family = FONT_FAMILY, size = FONT$annotation/.pt) + scale_fill_manual(values = c(GTEx_Normal = COHORT_COLOURS[["GTEx_Normal"]],
    `TCGA-PAAD_Tumor` = COHORT_COLOURS[["TCGA-PAAD_Tumor"]]), guide = "none") + scale_x_discrete(labels = label_cohort) +
    scale_y_continuous(limits = c(0, 1), expand = expansion(mult = c(0, 0.08))) + labs(x = NULL, y = expression("Axial " *
    R[2])) + publication_theme()



#### Supplementary Figure 5E–G: CPTAC and three-cohort membership


In [ ]:
# Cell 9: CPTAC overlap, Venn, and permutation analysis
CPTACoverlap <- ov2(sets$GTEx_Normal, sets[["CPTAC-3_Tumor"]], "GTEx", "CPTAC")
write.csv(CPTACoverlap, file.path(TABLE_DIR, "s5_e_gtex_cptac_overlap.csv"), row.names = FALSE)
s5_e <- overlap_bar(CPTACoverlap, "GTEx_Normal", "CPTAC-3_Tumor")
a <- sets$GTEx_Normal
b <- sets[["TCGA-PAAD_Tumor"]]
c <- sets[["CPTAC-3_Tumor"]]
threevenn <- data.frame(region = c("GTEx only", "TCGA only", "CPTAC only", "GTEx + TCGA", "GTEx + CPTAC",
    "TCGA + CPTAC", "All three"), n_genes = c(length(setdiff(a, union(b, c))), length(setdiff(b, union(a,
    c))), length(setdiff(c, union(a, b))), length(setdiff(intersect(a, b), c)), length(setdiff(intersect(a,
    c), b)), length(setdiff(intersect(b, c), a)), length(Reduce(intersect, list(a, b, c)))))
write.csv(threevenn, file.path(TABLE_DIR, "s5_f_three_cohort_venn_regions.csv"), row.names = FALSE)
cc <- rbind(circle(-0.53, 0.22, 0.82, "GTEx"), circle(0.53, 0.22, 0.82, "TCGA"), circle(0, -0.48, 0.82,
    "CPTAC"))
p3 <- data.frame(x = c(-0.8, 0.8, 0, -0.25, -0.42, 0.42, 0), y = c(0.22, 0.22, -0.72, 0.47, -0.27, -0.27,
    0.03), label = threevenn$n_genes)
s5_f <- ggplot() + geom_polygon(data = cc, aes(x, y, group = group, fill = group), alpha = 0.25, colour = NA) +
    geom_path(data = cc, aes(x, y, group = group, colour = group), linewidth = 0.4) + geom_text(data = p3,
    aes(x, y, label = label), family = FONT_FAMILY, size = FONT$annotation/.pt, fontface = "bold") +
    annotate("text", x = c(-0.78, 0.78, 0), y = c(1.15, 1.15, -1.35), label = c("GTEx", "TCGA", "CPTAC"),
        colour = c(COHORT_COLOURS[["GTEx_Normal"]], COHORT_COLOURS[["TCGA-PAAD_Tumor"]], COHORT_COLOURS[["CPTAC-3_Tumor"]]),
        family = FONT_FAMILY, size = FONT$legend/.pt) + scale_fill_manual(values = c(GTEx = COHORT_COLOURS[["GTEx_Normal"]],
    TCGA = COHORT_COLOURS[["TCGA-PAAD_Tumor"]], CPTAC = COHORT_COLOURS[["CPTAC-3_Tumor"]]), guide = "none") +
    scale_colour_manual(values = c(GTEx = COHORT_COLOURS[["GTEx_Normal"]], TCGA = COHORT_COLOURS[["TCGA-PAAD_Tumor"]],
        CPTAC = COHORT_COLOURS[["CPTAC-3_Tumor"]]), guide = "none") + coord_fixed(xlim = c(-1.5, 1.5),
    ylim = c(-1.55, 1.45), clip = "off") + publication_theme() + theme(axis.text = element_blank(), axis.title = element_blank(),
    panel.grid = element_blank())
pd <- perm_delta(gtex$acrophase, tcga$acrophase, 20000L, 2001L)
delta_qc <- data.frame(comparison = "TCGA minus GTEx", n_GTEx = nrow(gtex), n_TCGA = nrow(tcga), delta_R2 = -pd$delta,
    p_value = pd$p, permutations = 20000L, seed = 2001L)
write.csv(delta_qc, file.path(TABLE_DIR, "s5_g_delta_axial_permutation.csv"), row.names = FALSE)
s5_g <- ggplot(data.frame(delta_R2 = -pd$null), aes(delta_R2)) + geom_histogram(bins = 45, fill = "grey75",
    colour = "white", linewidth = 0.15) + geom_vline(xintercept = delta_qc$delta_R2, colour = COHORT_COLOURS[["TCGA-PAAD_Tumor"]],
    linewidth = 0.55) + annotate("text", x = delta_qc$delta_R2, y = Inf, parse = TRUE, label = sprintf("atop('observed = %.3f', italic(p)==%.3g)",
    delta_qc$delta_R2, delta_qc$p_value), hjust = 1.05, vjust = 1.2, family = FONT_FAMILY, size = FONT$annotation/.pt) +
    labs(x = expression(Delta * " axial " * R[2] * " (TCGA - GTEx)"), y = "Permutations") + publication_theme()



#### Supplementary Figure 6: common versus GTEx-unique tumor genes


In [ ]:
# Cell 10: Common-versus-unique comparisons
tumors <- c("TCGA-PAAD_Tumor", "CPTAC-3_Tumor")
cu <- do.call(rbind, lapply(tumors, function(nm) {
    x <- sig[[nm]]
    x$Category <- ifelse(x$Gene %in% gtex$Gene, "Common with GTEx", "Unique to tumor")
    x
}))
# Display colours identify the reference-shared group and the cohort-specific unique group.
# `dodge_group` retains two equally spaced groups within each tumor cohort.
cu$display_group <- ifelse(cu$Category == "Common with GTEx", "Common with GTEx",
    paste0(label_cohort(cu$cohort), " unique"))
cu$display_group <- factor(cu$display_group, levels = c("Common with GTEx", "TCGA unique", "CPTAC unique"))
cu$dodge_group <- factor(ifelse(cu$Category == "Common with GTEx", "Common", "Unique"),
    levels = c("Common", "Unique"))
cu$neg_log10_p <- -log10(pmax(cu$p, .Machine$double.xmin))
s6_common_unique_colours <- c(`Common with GTEx` = COHORT_COLOURS[["GTEx_Normal"]],
    `TCGA unique` = COHORT_COLOURS[["TCGA-PAAD_Tumor"]], `CPTAC unique` = COHORT_COLOURS[["CPTAC-3_Tumor"]])
pvalue_qc <- do.call(rbind, lapply(split(cu, cu$cohort), function(x) {
    a <- x$neg_log10_p[x$Category == "Common with GTEx"]
    b <- x$neg_log10_p[x$Category == "Unique to tumor"]
    data.frame(cohort = x$cohort[1], n_common = length(a), n_unique = length(b), cohens_d = cohens_d(a,
        b))
}))
write.csv(pvalue_qc, file.path(TABLE_DIR, "s6_a_pvalue_effect_sizes.csv"), row.names = FALSE)
s6_a <- ggplot(cu, aes(cohort, neg_log10_p, fill = display_group,
    group = interaction(cohort, dodge_group))) +
    geom_violin(position = position_dodge(0.72), width = 0.7, alpha = 0.7, linewidth = 0.25) +
    geom_boxplot(position = position_dodge(0.72), width = 0.12, outlier.shape = NA, linewidth = 0.25) +
    geom_text(data = pvalue_qc, aes(x = cohort, y = Inf, label = sprintf("d = %.3f", cohens_d)),
        inherit.aes = FALSE, vjust = 1.25, family = FONT_FAMILY, size = FONT$annotation/.pt) +
    scale_fill_manual(values = s6_common_unique_colours) + scale_x_discrete(labels = label_cohort) +
    labs(x = NULL, y = expression(-log[10](italic(p))), fill = NULL) + guides(fill = guide_legend(ncol = 1)) +
    publication_theme() + theme(legend.position = "bottom")
dc <- do.call(rbind, lapply(split(cu, interaction(cu$cohort, cu$Category, drop = TRUE)), function(x) {
    d <- legacy_density(x$acrophase)
    d$cohort <- x$cohort[1]
    d$Category <- x$Category[1]
    d$display_group <- x$display_group[1]
    d
}))
wq <- do.call(rbind, lapply(split(dc, dc$cohort), function(x) {
    z <- x[x$Category == "Common with GTEx", ]
    u <- x[x$Category == "Unique to tumor", ]
    data.frame(cohort = x$cohort[1], W1_hours = w1(z$theta, z$density, u$density) * 24/(2 * pi))
}))
write.csv(dc, file.path(TABLE_DIR, "s6_b_common_unique_density.csv"), row.names = FALSE)
write.csv(wq, file.path(TABLE_DIR, "s6_b_circular_w1.csv"), row.names = FALSE)
s6_b <- ggplot(dc, aes(theta, density, colour = display_group)) + geom_line(linewidth = 0.42) +
    coord_polar(theta = "x") + scale_x_continuous(breaks = seq(0, 2 * pi, pi/3),
    labels = c("0", "4", "8", "12", "16", "20", "24"), expand = c(0, 0)) +
    scale_colour_manual(values = s6_common_unique_colours) + facet_wrap(~cohort, ncol = 1,
    labeller = labeller(cohort = label_cohort)) + geom_text(data = wq, aes(x = pi, y = Inf,
    label = sprintf("W1 = %.2f h", W1_hours)), inherit.aes = FALSE, vjust = 1.25, family = FONT_FAMILY,
    size = FONT$annotation/.pt) + labs(x = NULL, y = "Density", colour = NULL) +
    guides(colour = guide_legend(ncol = 1)) + publication_theme() + theme(legend.position = "bottom")
cu$peak_to_trough <- 2 * cu$Amplitude + 1
ksq <- do.call(rbind, lapply(split(cu, cu$cohort), function(x) {
    a <- x$peak_to_trough[x$Category == "Common with GTEx"]
    b <- x$peak_to_trough[x$Category == "Unique to tumor"]
    k <- suppressWarnings(ks.test(a, b, exact = FALSE))
    data.frame(cohort = x$cohort[1], n_common = length(a), n_unique = length(b), KS_D = unname(k$statistic),
        KS_p = k$p.value)
}))
acc <- do.call(rbind, lapply(split(cu, interaction(cu$cohort, cu$Category, drop = TRUE)), function(x) {
    q <- seq(min(x$peak_to_trough), max(x$peak_to_trough), length.out = 200)
    data.frame(log2_peak_to_trough = log2(q), n_genes = vapply(q, function(z) sum(x$peak_to_trough >=
        z), integer(1)), cohort = x$cohort[1], Category = x$Category[1], display_group = x$display_group[1])
}))
write.csv(ksq, file.path(TABLE_DIR, "s6_c_amplitude_ks.csv"), row.names = FALSE)
write.csv(acc, file.path(TABLE_DIR, "s6_c_amplitude_accumulation.csv"), row.names = FALSE)
s6_c <- ggplot(acc, aes(log2_peak_to_trough, n_genes, colour = display_group)) + geom_line(linewidth = 0.42) +
    scale_y_log10() + scale_colour_manual(values = s6_common_unique_colours) + facet_wrap(~cohort,
    ncol = 2, scales = "free_y", labeller = labeller(cohort = label_cohort)) + geom_text(data = ksq,
    aes(x = Inf, y = Inf, label = sprintf("atop('D = %.3f', italic(p) == '%.2g')", KS_D, KS_p)), inherit.aes = FALSE, parse = TRUE,
    hjust = 1.08, vjust = 1.15, family = FONT_FAMILY, size = FONT$annotation/.pt) + labs(x = expression(log[2] *
    "(2 " * Amplitude + " + 1)"), y = "Genes at or above threshold", colour = NULL) + publication_theme() +
    theme(legend.position = "bottom")


At a 52.5-mm module: Figure 2H is 1 × 2; I–J are 1 × 1. K and L occupy a shared 2 × 1 block, split into heights 0.6 and 0.4 modules (31.5 and 21 mm). The assembled Figure 2 is 4 × 5 modules. S5E is 1 × 1, S5F 2 × 1, S5G 1 × 1; S6A 1 × 2, S6B 1 × 2 (vertical cohorts), and S6C 2 × 2. PDF panel letters and PNG omissions follow the project conventions.


### Figure 2K–L and Supplementary Figure 5J–K: ChIP and CCG PPI enrichment

**Shared target definition.** A rhythmic gene passes cohort-wise BH q < 0.05 and peak-to-trough amplitude > 0.1 (log2 CPM). Current all-gene regression tables define every enrichment target. A tumor-specific gene is rhythmic in that tumor cohort, tested with finite q and amplitude in both GTEx and CPTAC normal cohorts, and does not pass the rhythmicity rule in either normal cohort. This is an operational threshold definition, not proof of absent normal rhythms or a direct differential-rhythmicity test. It is distinct from the pairwise tumor-only regions in the overlap plots. The same current TCGA-specific ENSG set feeds K, L, and S5J–K; the same procedure defines CPTAC-specific targets. Gene-level selection/mapping and threshold QC are exported.

Figure 2K uses the 16,570 shared tested, promoter-assessed protein-coding genes (GENCODE v49) as its background. Binding means at least one ChIP-Atlas peak with score >=200 overlapping a GRCh38 promoter (TSS +/-2 kb); assessed zero-binding genes remain in the background. The pooled endpoint is the union across nine TFs. Upper-tail hypergeometric tests are independently checked against one-sided Fisher exact tests. BH adjustment covers all 20 cohort/endpoint tests. Numbers show fold enrichment; colors and stars use adjusted q (* q<0.05, ** q<0.01, *** q<0.001).

Figure 2L and S5J–K recompute membership and enrichment from these current targets using the processed BioGRID physical-interaction symbol background (22,695 symbols) and the original ten-clock-gene partner sets. GTEx uses all current rhythmic genes; both tumor cohorts use their current tumor-specific sets. The pooled set contains 633 unique partners. ENSG identifiers map through GENCODE v49 to uppercase symbols, are deduplicated, and intersect the PPI background. PPI retains raw-p plotting conventions and its broader legacy network universe (no protein-coding/shared-tested restriction); differences in eligible target counts between assays are therefore expected. Hypergeometric tests are independently checked against Fisher exact tests. These enrichment tests do not directly compare cohorts.

Processed peak-assignment, PPI partner, and PPI background-membership products must accompany a public release. The notebook recomputes every enrichment count and test without loading saved enrichment statistics or saved cancer-specific gene lists. It does not repeat raw peak-to-promoter assignment or BioGRID parsing.


In [ ]:
# Cell 11: Current rhythmic targets and recomputed ChIP/PPI enrichment
library(dplyr)
library(tidyr)
library(readr)
library(ggplot2)
library(stringr)

ppi_dir <- file.path(DATA_DIR, "enrichment", "ppi_enrichment")
ppi_partners_path <- file.path(ppi_dir, "clock_gene_ppi_partners.txt")
stopifnot(file.exists(ppi_partners_path))

cohort_key <- c(GTEx_Normal = "GTEx", `TCGA-PAAD_Tumor` = "TCGA", `CPTAC-3_Tumor` = "CPTAC")
cohort_order <- names(cohort_key)
clock_tf_order <- c("ARNTL", "CLOCK", "CRY1", "NPAS2", "NR1D1", "RORA", "RORC", "DBP", "TEF")
clock_tf_labels <- c(ARNTL = "BMAL1", CLOCK = "CLOCK", CRY1 = "CRY1", DBP = "DBP", NPAS2 = "NPAS2", NR1D1 = "NR1D1",
    RORA = "RORA", RORC = "RORC", TEF = "TEF")
ppi_clock_order <- c("ARNTL", "PER1", "PER2", "PER3", "CRY1", "CRY2", "NR1D1", "NR1D2", "CIART", "NPAS2")
ppi_clock_labels <- c(ARNTL = "BMAL1", PER1 = "PER1", PER2 = "PER2", PER3 = "PER3", CRY1 = "CRY1", CRY2 = "CRY2",
    NR1D1 = "NR1D1", NR1D2 = "NR1D2", CIART = "CIART", NPAS2 = "NPAS2")
significance_stars <- function(p) dplyr::case_when(p < 0.001 ~ "***", p < 0.01 ~ "**", p < 0.05 ~ "*", TRUE ~
    "")
short_cohort <- function(x) unname(cohort_key[x])

# One current rhythmicity rule across all cohorts and enrichment targets.
clean_ensg <- function(x) str_remove(as.character(x), "\\..*$")
regression_cohorts <- c("GTEx_Normal", "CPTAC-3_Normal", "TCGA-PAAD_Tumor", "CPTAC-3_Tumor")
current_regression_paths <- setNames(file.path(DATA_DIR, "Merged", "regression",
    paste0("all_genes_regression_periodic_12h_", regression_cohorts, ".txt")), regression_cohorts)
current_regression <- lapply(current_regression_paths, function(path) {
    d <- read_tsv(path, show_col_types = FALSE) |>
        mutate(ENSG = clean_ensg(ENSG), tested = is.finite(q) & is.finite(Amplitude),
            rhythmic = tested & q < RHYTHM_Q & Amplitude > RHYTHM_AMPLITUDE)
    stopifnot(!anyDuplicated(d$ENSG))
    d
})
current_rhythmic_ids <- lapply(current_regression, function(d) d$ENSG[d$rhythmic])
for (g in COHORTS) stopifnot(setequal(current_rhythmic_ids[[g]], clean_ensg(sig_tables[[g]]$ENSG)))
normal_cohorts <- c("GTEx_Normal", "CPTAC-3_Normal")
normal_tested <- Reduce(intersect, lapply(current_regression[normal_cohorts], function(d) d$ENSG[d$tested]))
normal_rhythmic <- Reduce(union, current_rhythmic_ids[normal_cohorts])
tumor_groups <- c("TCGA-PAAD_Tumor", "CPTAC-3_Tumor")
current_specific_ids <- setNames(lapply(tumor_groups, function(g)
    setdiff(intersect(current_rhythmic_ids[[g]], normal_tested), normal_rhythmic)), tumor_groups)
# Failure to meet the rhythmicity threshold is an operational definition, not evidence of no rhythm.
current_specific_membership <- bind_rows(lapply(tumor_groups, function(g) {
    d <- current_regression[[g]] |>
        transmute(group = g, ENSG, Gene, tumor_q = q, tumor_amplitude = Amplitude, tumor_rhythmic = rhythmic)
    for (normal in normal_cohorts) {
        nd <- current_regression[[normal]] |>
            select(ENSG, q, Amplitude, tested, rhythmic)
        names(nd)[-1] <- paste0(normal, "_", names(nd)[-1])
        d <- left_join(d, nd, by = "ENSG")
    }
    d |> mutate(tested_in_both_normals = ENSG %in% normal_tested,
        rhythmic_in_any_normal = ENSG %in% normal_rhythmic,
        tumor_specific = ENSG %in% current_specific_ids[[g]])
}))
current_definition_qc <- bind_rows(lapply(regression_cohorts, function(g) {
    d <- current_regression[[g]]
    tibble(group = g, tested = sum(d$tested), rhythmic = sum(d$rhythmic),
        q_threshold = RHYTHM_Q, peak_to_trough_threshold = RHYTHM_AMPLITUDE,
        tumor_specific = if (g %in% tumor_groups) length(current_specific_ids[[g]]) else NA_integer_)
}))
stopifnot(all(vapply(tumor_groups, function(g) all(current_specific_ids[[g]] %in% current_rhythmic_ids[[g]]), logical(1))))
write_tsv(current_specific_membership, file.path(TABLE_DIR, "current_tumor_specific_membership.tsv"))
write_tsv(current_definition_qc, file.path(TABLE_DIR, "current_rhythmic_definition_qc.tsv"))
print(current_definition_qc)

tf_order <- c("ARNTL", "CLOCK", "CRY1", "DBP", "NPAS2", "NR1D1", "RORA", "RORC", "TEF")
endpoint_order <- c(tf_order, "ANY_CCG_PEAK")

data_path <- function(...) file.path(as.character(DATA_DIR), ...)
table_path <- function(...) file.path(as.character(TABLE_DIR), ...)
chip_input_paths <- c(
  assessed_promoter_peak_counts = data_path("chip_background_sensitivity", "assessed_promoter_peak_counts.tsv"),
  eligible_gene_membership = data_path("chip_background_sensitivity", "eligible_gene_membership.tsv"),
  gencode_v49_coordinates = data_path("reference", "gencode_gene_coordinates.tsv"),
  gtex_significant_rhythmic = data_path("Merged", "regression", "sig_genes_regression_periodic_12h_GTEx_Normal.txt")
)

clean_ensg <- function(x) str_remove(as.character(x), "\\..*$")
id_column <- function(x) {
  candidates <- c("ENSG_clean", "ENSG", "gene_id", "Gene")
  hit <- candidates[candidates %in% names(x)]
  stopifnot(length(hit) > 0L)
  hit[[1]]
}

peak_counts <- read_tsv(chip_input_paths[["assessed_promoter_peak_counts"]], show_col_types = FALSE) |>
  mutate(ENSG = clean_ensg(ENSG))
eligible_membership <- read_tsv(chip_input_paths[["eligible_gene_membership"]], show_col_types = FALSE) |>
  mutate(ENSG = clean_ensg(ENSG))
gencode_v49 <- read_tsv(chip_input_paths[["gencode_v49_coordinates"]], show_col_types = FALSE) |>
  mutate(ENSG = clean_ensg(gene_id))
stopifnot(all(tf_order %in% names(peak_counts)), !anyDuplicated(peak_counts$ENSG),
          !anyDuplicated(eligible_membership$ENSG), !anyDuplicated(gencode_v49$ENSG))
stopifnot(all(vapply(peak_counts[tf_order], function(x) all(is.finite(x) & x >= 0 & x == as.integer(x)), logical(1))))

gtex_sig <- sig_tables[["GTEx_Normal"]]
gtex_ids <- unique(clean_ensg(gtex_sig[[id_column(gtex_sig)]]))
tcga_ids <- current_specific_ids[["TCGA-PAAD_Tumor"]]
protein_coding <- unique(gencode_v49$ENSG[gencode_v49$gene_type == "protein_coding"])
shared_background <- unique(eligible_membership$ENSG[toupper(as.character(eligible_membership$shared_background)) == "TRUE"])
universe <- intersect(shared_background, protein_coding)

stopifnot(length(universe) == 16570L)
stopifnot(all(universe %in% peak_counts$ENSG))
targets <- list(
  GTEx_Normal = intersect(gtex_ids, universe),
  `TCGA-PAAD_Tumor` = intersect(intersect(tcga_ids, protein_coding), universe)
)
stopifnot(length(targets[["GTEx_Normal"]]) == 8076L,
    all(tcga_ids %in% current_rhythmic_ids[["TCGA-PAAD_Tumor"]]))

bound_sets <- lapply(tf_order, function(tf) unique(peak_counts$ENSG[peak_counts[[tf]] > 0L]))
names(bound_sets) <- tf_order
bound_sets[["ANY_CCG_PEAK"]] <- unique(unlist(bound_sets, use.names = FALSE))

chip_summary <- bind_rows(lapply(names(targets), function(group) {
  target <- targets[[group]]
  bind_rows(lapply(endpoint_order, function(tf) {
    bound <- intersect(bound_sets[[tf]], universe)
    M <- length(universe); n <- length(bound); N <- length(target); k <- length(intersect(target, bound))
    remainder_positive <- n - k
    remainder_negative <- M - N - n + k
    stopifnot(all(c(k, N - k, remainder_positive, remainder_negative) >= 0L))
    pvalue <- phyper(k - 1L, n, M - n, N, lower.tail = FALSE)
    fisher_pvalue <- fisher.test(matrix(c(k, N - k, remainder_positive, remainder_negative), nrow = 2, byrow = TRUE), alternative = "greater")$p.value
    stopifnot(isTRUE(all.equal(pvalue, fisher_pvalue, tolerance = 1e-10)))
    tibble(group = group, tf = tf, M = M, n = n, N = N, k = k,
           fold_enrichment = (k / N) / (n / M), pvalue = pvalue, fisher_pvalue = fisher_pvalue,
           target_positive = k, target_negative = N - k,
           remainder_positive = remainder_positive, remainder_negative = remainder_negative)
  }))
})) |>
  mutate(qvalue = p.adjust(pvalue, method = "BH"), tf = factor(tf, levels = endpoint_order)) |>
  arrange(group, tf) |>
  mutate(tf = as.character(tf))

tcga_all_rhythmic <- unique(clean_ensg(sig_tables[["TCGA-PAAD_Tumor"]][[id_column(sig_tables[["TCGA-PAAD_Tumor"]])]]))
chip_qc <- tibble(
  group = c("GTEx_Normal", "TCGA-PAAD_Tumor"),
  raw_target_genes = c(length(gtex_ids), length(tcga_ids)),
  protein_coding_target_genes = c(length(intersect(gtex_ids, protein_coding)), length(intersect(tcga_ids, protein_coding))),
  target_in_shared_background = c(length(targets[["GTEx_Normal"]]), length(targets[["TCGA-PAAD_Tumor"]])),
  shared_background_protein_coding = length(universe),
  tcga_absent_from_current_rhythmic = c(NA_integer_, length(setdiff(tcga_ids, tcga_all_rhythmic)))
)
stopifnot(chip_qc$tcga_absent_from_current_rhythmic[[2]] == 0L)

target_membership <- bind_rows(lapply(names(targets), function(group) {
  candidate <- if (group == "GTEx_Normal") gtex_ids else tcga_ids
  target_ids <- targets[[group]]
  tibble(group = group, ENSG = sort(unique(candidate))) |>
    mutate(protein_coding = ENSG %in% protein_coding,
           shared_all_tested = ENSG %in% shared_background,
           in_target = ENSG %in% target_ids)
}))

chip_qc_summary <- chip_summary |>
  transmute(group, tf, M, n, N, k, pvalue, fisher_pvalue,
            fisher_abs_delta = abs(pvalue - fisher_pvalue))
stopifnot(all(chip_qc_summary$fisher_abs_delta <= 1e-10))
write_tsv(chip_summary, table_path("fig2k_chip_enrichment_source.tsv"))
write_tsv(chip_qc, table_path("fig2k_chip_enrichment_qc.tsv"))
write_tsv(chip_qc_summary, table_path("fig2k_chip_enrichment_qc_summary.tsv"))
write_tsv(target_membership, table_path("fig2k_chip_target_membership.tsv"))

cat(sprintf("Figure 2K QC: M=%d; GTEx N=%d; TCGA raw=%d, protein_coding/in_background N=%d, absent_from_current_rhythmic=%d; pooled n=%d, k=(GTEx %d, TCGA %d)\n",
            length(universe), length(targets[["GTEx_Normal"]]), length(tcga_ids),
            length(targets[["TCGA-PAAD_Tumor"]]), chip_qc$tcga_absent_from_current_rhythmic[[2]],
            chip_summary$n[chip_summary$group == "GTEx_Normal" & chip_summary$tf == "ANY_CCG_PEAK"],
            chip_summary$k[chip_summary$group == "GTEx_Normal" & chip_summary$tf == "ANY_CCG_PEAK"],
            chip_summary$k[chip_summary$group == "TCGA-PAAD_Tumor" & chip_summary$tf == "ANY_CCG_PEAK"]))
print(chip_qc)
print(chip_summary |> transmute(group, tf = dplyr::recode(tf, ARNTL = "BMAL1"), M, n, N, k, fold_enrichment, pvalue, qvalue))

chip_endpoint_order <- c(clock_tf_order, "ANY_CCG_PEAK")
chip_endpoint_labels <- c(clock_tf_labels, ANY_CCG_PEAK = "Pooled CCG\npeaks")
chip_group_labels <- c(GTEx_Normal = "GTEx", `TCGA-PAAD_Tumor` = "TCGA-\nspecific")
chip_plot <- chip_summary %>%
    mutate(tf = factor(tf, levels = chip_endpoint_order),
           group = factor(group, levels = rev(c("GTEx_Normal", "TCGA-PAAD_Tumor"))),
           neg_log10_q = -log10(pmax(qvalue, .Machine$double.xmin)),
           star = significance_stars(qvalue),
           label = if_else(star == "", sprintf("%.3f", fold_enrichment),
                           paste0(sprintf("%.3f", fold_enrichment), "\n", star)))
# Use actual scale colours for readable black/white annotations.
chip_fill_colours <- scales::col_numeric(c("white", "#B2182B"),
    domain = c(0, max(chip_plot$neg_log10_q)))(chip_plot$neg_log10_q)
chip_rgb <- grDevices::col2rgb(chip_fill_colours) / 255
chip_luminance <- as.vector(c(0.2126, 0.7152, 0.0722) %*%
    ifelse(chip_rgb <= 0.04045, chip_rgb / 12.92, ((chip_rgb + 0.055) / 1.055)^2.4))
chip_plot$text_colour <- ifelse(chip_luminance < 0.179, "white", "black")
fig2_k <- ggplot(chip_plot, aes(tf, group, fill = neg_log10_q)) +
    geom_tile(colour = "white", linewidth = 0.25) +
    geom_text(aes(label = label, colour = text_colour), family = FONT_FAMILY,
        size = FONT$annotation/ggplot2::.pt, lineheight = 0.8) +
    scale_colour_identity() +
    scale_x_discrete(labels = chip_endpoint_labels, expand = expansion(add = 0)) +
    scale_y_discrete(labels = chip_group_labels, expand = expansion(add = 0)) +
    scale_fill_gradient(low = "white", high = "#B2182B", name = expression("-log10(" * italic(q) * ")"),
        limits = c(0, max(chip_plot$neg_log10_q))) +
    labs(title = "Clock TF ChIP binding enrichment", x = NULL, y = NULL,
        subtitle = expression(paste("Fold enrichment; * ", italic(q), "<0.05, ** ", italic(q), "<0.01, *** ", italic(q), "<0.001"))) +
    publication_theme() +
    theme(axis.text.x = element_text(size = FONT$tick, angle = 45, hjust = 1),
        axis.text.y = element_text(size = FONT$tick), axis.line = element_blank(),
        axis.ticks = element_blank(), legend.position = "bottom", legend.direction = "horizontal",
        legend.margin = margin(0, 0, 0, 0), legend.box.spacing = grid::unit(0, "mm"),
        plot.margin = margin(2, 1, 1, 1, "mm"), panel.grid = element_blank()) +
    guides(fill = guide_colourbar(title.position = "left", barwidth = grid::unit(24, "mm"),
        barheight = grid::unit(2.5, "mm")))

# Recompute PPI membership and statistics from the processed network universe/partner sets.
ppi_background_path <- file.path(DATA_DIR, "ppi_panel_l_background_comparison", "background_symbols.tsv")
ppi_background <- unique(toupper(read_tsv(ppi_background_path, show_col_types = FALSE)$symbol))
ppi_partners <- read_tsv(ppi_partners_path, show_col_types = FALSE)
stopifnot(length(ppi_background) == 22695L, all(ppi_clock_order %in% ppi_partners$Clock_Gene))
ppi_partner_sets <- setNames(lapply(ppi_clock_order, function(g) {
    values <- unique(toupper(unlist(strsplit(ppi_partners$Partners[ppi_partners$Clock_Gene == g], ",", fixed = TRUE))))
    intersect(setdiff(values, g), ppi_background)
}), ppi_clock_order)
ppi_partner_sets$combined <- unique(unlist(ppi_partner_sets, use.names = FALSE))
stopifnot(length(ppi_partner_sets$combined) == 633L)
raw_ppi_targets <- c(list(GTEx_Normal = current_rhythmic_ids[["GTEx_Normal"]]), current_specific_ids)
ppi_target_mapping <- bind_rows(lapply(names(raw_ppi_targets), function(g) {
    tibble(group = g, ENSG = raw_ppi_targets[[g]]) |>
        left_join(gencode_v49 |> select(ENSG, gene_name), by = "ENSG") |>
        mutate(symbol = toupper(gene_name), in_background = !is.na(symbol) & symbol %in% ppi_background)
}))
ppi_targets <- setNames(lapply(cohort_order, function(g)
    unique(ppi_target_mapping$symbol[ppi_target_mapping$group == g & ppi_target_mapping$in_background])), cohort_order)
ppi_tests <- bind_rows(lapply(cohort_order, function(g) {
    target <- ppi_targets[[g]]
    bind_rows(lapply(names(ppi_partner_sets), function(clock) {
        partner <- ppi_partner_sets[[clock]]
        M <- length(ppi_background); n <- length(partner); N <- length(target); k <- length(intersect(target, partner))
        ct <- c(k, N-k, n-k, M-N-n+k)
        stopifnot(all(ct >= 0))
        p <- phyper(k-1L, n, M-n, N, lower.tail = FALSE)
        fp <- fisher.test(matrix(ct, nrow = 2, byrow = TRUE), alternative = "greater")$p.value
        stopifnot(isTRUE(all.equal(p, fp, tolerance = 1e-10)))
        tibble(group = g, clock_gene = clock, M, n, N, k,
            fold_enrichment = (k/N)/(n/M), pvalue = p, fisher_pvalue = fp,
            target_positive = k, target_negative = N-k, remainder_positive = n-k, remainder_negative = M-N-n+k)
    }))
}))
ppi_combined <- ppi_tests |> filter(clock_gene == "combined") |>
    mutate(group = factor(group, levels = cohort_order), group_label = short_cohort(as.character(group)), star = significance_stars(pvalue))
ppi_per_clock <- ppi_tests |> filter(clock_gene != "combined") |>
    mutate(group = factor(group, levels = c("TCGA-PAAD_Tumor", "GTEx_Normal", "CPTAC-3_Tumor")),
        clock_gene = factor(clock_gene, levels = ppi_clock_order), neg_log10_p = -log10(pmax(pvalue, .Machine$double.xmin)))
ppi_qc <- ppi_tests |>
    mutate(expected = N*n/M, fold_recomputed = k/expected,
        fold_delta = fold_enrichment-fold_recomputed, p_delta = pvalue-fisher_pvalue)
ppi_qc_magnitude <- ppi_qc |> summarise(max_abs_fold_delta = max(abs(fold_delta)), max_abs_p_delta = max(abs(p_delta)))
stopifnot(ppi_qc_magnitude$max_abs_fold_delta < 1e-10, ppi_qc_magnitude$max_abs_p_delta < 1e-10)
write_tsv(ppi_target_mapping, file.path(TABLE_DIR, "fig2l_s5jk_ppi_target_membership.tsv"))
write_tsv(ppi_combined, file.path(TABLE_DIR, "fig2l_ccg_ppi_combined_source.tsv"))
write_tsv(ppi_per_clock, file.path(TABLE_DIR, "s5jk_ccg_ppi_per_clock_source.tsv"))
write_tsv(ppi_qc, file.path(TABLE_DIR, "fig2l_s5jk_ccg_ppi_qc.tsv"))
write_tsv(ppi_qc_magnitude, file.path(TABLE_DIR, "fig2l_s5jk_ccg_ppi_qc_summary.tsv"))
print(ppi_combined |> select(group, M, n, N, k, fold_enrichment, pvalue))

ppi_two <- ppi_combined %>%
    filter(group %in% c("GTEx_Normal", "TCGA-PAAD_Tumor")) %>%
    mutate(group = factor(group, levels = rev(c("GTEx_Normal", "TCGA-PAAD_Tumor"))), group_label = short_cohort(as.character(group)))
# Panel L and S5 use exactly the current tumor-specific ENSG targets defined above.
fig2_l <- ggplot(ppi_two, aes(fold_enrichment, group, fill = group_label)) + geom_col(width = 0.68, colour = "black",
    linewidth = 0.2) + geom_vline(xintercept = 1, linetype = "dashed", linewidth = 0.3) + geom_text(aes(label = star,
    x = fold_enrichment + 0.035), hjust = 0, family = FONT_FAMILY, size = FONT$annotation/ggplot2::.pt) +
    scale_fill_manual(values = COHORT_COLOURS[c("GTEx_Normal", "TCGA-PAAD_Tumor")], guide = "none") +
    scale_y_discrete(labels = c(GTEx_Normal = "GTEx", `TCGA-PAAD_Tumor` = "TCGA-\nspecific")) + coord_cartesian(clip = "off") + labs(title = "Rhythmic genes in CCG PPI network",
    x = "Fold enrichment", y = NULL) + publication_theme() + theme(plot.title = element_text(size = FONT$title,
    face = "bold"), axis.title.x = element_text(size = FONT$axis_title), axis.text = element_text(size = FONT$tick),
    plot.margin = margin(3, 12, 3, 3, "mm"), panel.grid.major.y = element_blank())

ppi_heatmap_base <- function(plot_data, value, label, fill_name, fill_scale) {
    ggplot(plot_data, aes(clock_gene, group, fill = .data[[value]])) + geom_tile(colour = "white", linewidth = 0.25) +
        geom_text(aes(label = .data[[label]]), family = FONT_FAMILY, size = FONT$annotation/ggplot2::.pt) +
        scale_x_discrete(labels = ppi_clock_labels) + scale_y_discrete(labels = function(x) short_cohort(x)) +
        fill_scale + labs(x = "Clock gene", y = NULL, fill = fill_name) + publication_theme() + theme(axis.title.x = element_text(size = FONT$axis_title),
        axis.text.x = element_text(size = FONT$tick, angle = 45, hjust = 1), axis.text.y = element_text(size = FONT$tick),
        legend.title = element_text(size = FONT$legend), legend.text = element_text(size = FONT$legend),
        legend.position = "bottom", legend.direction = "horizontal", panel.grid = element_blank()) +
        guides(fill = guide_colourbar(barwidth = grid::unit(24, "mm"), barheight = grid::unit(2.5, "mm")))
}
s5_j_data <- ppi_per_clock %>%
    mutate(label_fe = sprintf("%.2f", fold_enrichment))
s5_k_data <- ppi_per_clock %>%
    mutate(label_logp = sprintf("%.1f", neg_log10_p))
s5_j <- ppi_heatmap_base(s5_j_data, "fold_enrichment", "label_fe", "Fold enrichment", scale_fill_gradient2(low = "#2166AC",
    mid = "white", high = "#B2182B", midpoint = 1)) + labs(title = "CCG PPI enrichment by clock gene")
s5_k <- ppi_heatmap_base(s5_k_data, "neg_log10_p", "label_logp", expression(-log[10](italic(p))), scale_fill_gradient(low = "white",
    high = "#B2182B")) + labs(title = "CCG PPI enrichment significance")

panel_size_mm <- c(width = 105, height = 52.5)  # 2 x 1 publication module.
# Parent integration adds panel tags and exports; all source/QC tables are written above.



## 6. Publication exports

Figure 2 retains the expression/polar rows, the heatmap/overlap row, and density/polarization alongside enrichment panels. S4 displays all clock-expression facets; S5 assembles cohort SVD, overlap, restricted-GTEx sensitivity, and PPI panels; S6 compares common and tumor-unique rhythmic genes. Panel titles use short cohort labels: GTEx = healthy pancreas, TCGA = TCGA-PAAD tumors, CPTAC = CPTAC-3 tumors.

Cohort colors are GTEx `#456BAB`, TCGA `#F2B770`, and CPTAC `#C46456` in cohort-coded main and supplementary plots. Panel K retains its red BH-q scale; other quantitative heatmap/significance scales remain distinct from cohort identity. In S6, common-with-GTEx genes are blue, TCGA-unique genes orange, and CPTAC-unique genes red.

Titles are centered. PDF panel letters overlay the upper-left margin without reserving a separate title row. B/D gene labels use the explicitly requested 5-pt exception and no leader lines. In K, lowercase italic q denotes the BH-adjusted q-value. Lowercase italic p denotes raw p-values.


In [ ]:
# Cell 12: Compact panel styling
clock_polar_frame <- function(d, phase_col, radius_col, title, radius_label, point_colour = NULL, fill_col = NULL, direct_labels = FALSE) {
    d <- as.data.frame(d)
    rmax <- max(d[[radius_col]]) * 1.08
    d$px <- (d[[radius_col]]/rmax) * sin(d[[phase_col]] * pi/12)
    d$py <- (d[[radius_col]]/rmax) * cos(d[[phase_col]] * pi/12)
    d$side <- ifelse(d$px >= 0, 1, -1)
    d$label_x <- d$side * 1.24
    d$label_y <- d$py
    for (side in c(-1, 1)) {
        ii <- which(d$side == side)
        ii <- ii[order(d$py[ii])]
        if (length(ii) > 1)
            d$label_y[ii] <- seq(-1.08, 1.08, length.out = length(ii))
    }
    d$label_hjust <- ifelse(d$side > 0, 0, 1)
    radial_values <- pretty(c(0, rmax), n = 3)
    radial_values <- radial_values[radial_values > 0 & radial_values <= rmax]
    theta_grid <- seq(0, 2 * pi, length.out = 181)
    rings <- bind_rows(lapply(radial_values, function(v) tibble(x = (v/rmax) * sin(theta_grid), y = (v/rmax) *
        cos(theta_grid), radius = v)))
    spokes <- tibble(hour = seq(0, 20, 4), x = sin(seq(0, 20, 4) * pi/12), y = cos(seq(0, 20, 4) * pi/12))
    ticks <- tibble(hour = c(0, 6, 12, 18), x = c(0, 0.96, 0, -0.96), y = c(1.12, 0, -1.12, 0), label = c("0/24",
        "6", "12", "18"))
    if (direct_labels) {
        ticks$x <- c(0, 1.10, 0, -1.10)
        ticks$y <- c(1.10, 0, -1.10, 0)
    }
    p <- ggplot() + geom_path(data = rings, aes(x, y, group = radius), colour = "#D9E0E5", linewidth = 0.2) +
        geom_segment(data = spokes, aes(x = 0, y = 0, xend = x, yend = y), colour = "#D9E0E5", linewidth = 0.2) +
        geom_text(data = ticks, aes(x, y, label = label), family = FONT_FAMILY,
        size = FONT$tick/ggplot2::.pt) + geom_text(data = tibble(x = 0, y = radial_values/rmax, label = format(radial_values,
        trim = TRUE)), aes(x, y, label = label), family = FONT_FAMILY, size = FONT$tick/ggplot2::.pt,
        hjust = 1.1, colour = "#59636B")
    if (is.null(fill_col))
        p <- p + geom_point(data = d, aes(px, py), colour = point_colour, size = 1.35) else {
        p <- p + geom_point(data = d, aes(px, py, fill = .data[[fill_col]]), shape = 21, colour = "#333333",
            stroke = 0.2, size = 1.35) + scale_fill_viridis_c(option = "magma", name = expression("-log10(" * italic(p) * ")"), limits = c(0,
            max(d[[fill_col]])), breaks = scales::breaks_pretty(n = 2)) + guides(fill = guide_colourbar(direction = "horizontal",
            title.position = "left", barwidth = grid::unit(18, "mm"), barheight = grid::unit(1.5, "mm")))
    }
    if (direct_labels) {
        # Invisible tick anchors prevent gene labels from colliding with hour labels.
        repel_data <- bind_rows(transmute(d, x = px, y = py, label = symbol),
            transmute(ticks, x, y, label = ""))
        p <- p + ggrepel::geom_text_repel(data = repel_data, aes(x, y, label = label),
            family = FONT_FAMILY, size = FONT$gene_annotation/ggplot2::.pt,
            seed = 213, segment.colour = NA, box.padding = 0.10, point.padding = 0.12,
            min.segment.length = Inf, max.overlaps = Inf, force = 0.5,
            max.iter = 20000, max.time = 2, xlim = c(-0.99, 0.99), ylim = c(-0.99, 0.99))
    } else {
        p <- p + geom_segment(data = d, aes(x = px, y = py, xend = label_x, yend = label_y),
            colour = "#89939B", linewidth = 0.2) +
            geom_text(data = d, aes(label_x, label_y, label = symbol, hjust = label_hjust),
            family = FONT_FAMILY, size = FONT$annotation/ggplot2::.pt)
    }
    if (direct_labels) {
        # Use the left/bottom axis-title positions and compact legends to enlarge the circular viewport.
        outer_ring <- tibble(x = sin(theta_grid), y = cos(theta_grid))
        p$layers <- c(list(geom_path(data = outer_ring, aes(x, y), colour = "#D9E0E5", linewidth = 0.2)), p$layers)
        p + coord_fixed(xlim = c(-1.20, 1.20), ylim = c(-1.20, 1.20), expand = FALSE, clip = "off") +
            labs(title = title, subtitle = NULL, x = "Phase (h)", y = "Amplitude") + publication_theme() +
            theme(axis.line = element_blank(), axis.ticks = element_blank(), axis.text = element_blank(),
                axis.title.x = element_text(size = FONT$axis_title, margin = margin(t = 0.5)),
                axis.title.y = element_text(size = FONT$axis_title, margin = margin(r = 0.5)),
                plot.title = element_text(size = FONT$title, face = "bold", hjust = 0.5, margin = margin(b = 1)),
                panel.grid = element_blank(), legend.position = "bottom", legend.margin = margin(0, 0, 0, 0),
                legend.box.spacing = grid::unit(0, "mm"), legend.spacing.y = grid::unit(0, "mm"),
                plot.margin = margin(0.8, 0.8, 0.6, 0.6, unit = "mm"))
    } else {
        p + coord_fixed(xlim = c(-2.08, 2.08), ylim = c(-1.25, 1.25), clip = "off") +
            labs(title = title, subtitle = radius_label, x = NULL, y = NULL) + publication_theme() +
            theme(axis.line = element_blank(), axis.ticks = element_blank(), axis.text = element_blank(),
                panel.grid = element_blank(), legend.position = "bottom", legend.margin = margin(0, 0, 0, 0),
                legend.box.spacing = grid::unit(0, "mm"), plot.margin = margin(1.8, 1.3, 1, 1.3, unit = "mm"))
    }
}
fig2_b <- clock_polar_frame(fig2_b$data, "acrophase", "Amplitude", "GTEx", "Peak-to-trough amplitude",
    fill_col = "neglog10p", direct_labels = TRUE)
fig2_d <- clock_polar_frame(fig2_d$data, "acrophase", "Amplitude", "TCGA", "Peak-to-trough amplitude",
    fill_col = "neglog10p", direct_labels = TRUE)
s5_d <- clock_polar_frame(s5_d$data, "acrophase", "Amplitude", "CPTAC", "Peak-to-trough amplitude", fill_col = "neglog10p")
s5_a <- clock_polar_frame(s5_a$data, "phase_h", "loading", "GTEx", "Mode 1 loading (100%)", COHORT_COLOURS[["GTEx_Normal"]])
s5_b <- clock_polar_frame(s5_b$data, "phase_h", "loading", "TCGA", "Mode 1 loading (100%)", COHORT_COLOURS[["TCGA-PAAD_Tumor"]])
s5_c <- clock_polar_frame(s5_c$data, "phase_h", "loading", "CPTAC", "Mode 1 loading (100%)", COHORT_COLOURS[["CPTAC-3_Tumor"]])
s5_h <- clock_polar_frame(s5_h$data, "phase_h", "loading", "GTEx ToD [0, 12)", "Mode 1 loading (100%)",
    COHORT_COLOURS[["GTEx_Normal"]])

# Reference and shared genes have the same fill; tumor-only genes occupy the top segment.
overlap_bar <- function(tumor, bar_width = 0.7) {
    reference <- sets[["GTEx_Normal"]]
    target <- sets[[tumor]]
    d <- tibble(cohort = c("GTEx", label_cohort(tumor), label_cohort(tumor)), category = factor(c("Shared / GTEx",
        "Shared / GTEx", "Tumor only"), levels = c("Tumor only", "Shared / GTEx")), n_genes = c(length(reference),
        length(intersect(reference, target)), length(setdiff(target, reference))))
    d$cohort <- factor(d$cohort, levels = c("GTEx", label_cohort(tumor)))
    ggplot(d, aes(cohort, n_genes, fill = category)) + geom_col(width = bar_width) + geom_text(aes(label = n_genes),
        position = position_stack(vjust = 0.5), family = FONT_FAMILY, size = FONT$annotation/ggplot2::.pt) +
        scale_fill_manual(values = c(`Shared / GTEx` = COHORT_COLOURS[["GTEx_Normal"]], `Tumor only` = COHORT_COLOURS[[tumor]]),
            breaks = c("Shared / GTEx", "Tumor only")) + scale_y_continuous(expand = expansion(mult = c(0,
        0.06))) + labs(x = NULL, y = "Rhythmic genes", fill = NULL) + guides(fill = guide_legend(ncol = 1,
        byrow = TRUE)) + publication_theme() + theme(legend.position = "bottom", legend.margin = margin(0,
        0, 0, 0), legend.key.height = grid::unit(2.5, "mm"), legend.key.width = grid::unit(3, "mm"),
        legend.box.spacing = grid::unit(0, "mm"), plot.margin = margin(2, 1, 1, 1, unit = "mm"))
}
fig2_h <- overlap_bar("TCGA-PAAD_Tumor", bar_width = 0.45)
s5_e <- overlap_bar("CPTAC-3_Tumor")
fig2_l <- fig2_l + aes(fill = group) + scale_fill_manual(values = COHORT_COLOURS, guide = "none")
s5_f <- s5_f + theme(axis.line = element_blank(), axis.ticks = element_blank())
s5_g <- s5_g + labs(x = expression(Delta * R[2]), y = "Permutations", title = "TCGA minus GTEx")
polar_grid_theme <- theme(panel.grid.major = element_line(colour = "#D9E0E5", linewidth = 0.2), axis.line = element_blank(),
    axis.ticks = element_blank())
fig2_i <- fig2_i + polar_grid_theme
s5_i <- s5_i + polar_grid_theme + labs(title = "GTEx ToD [0, 12): gene phases", y = "Density")
# Place Wasserstein statistics in facet strips instead of overlapping the 12 h tick.
s6_b$layers <- Filter(function(x) !inherits(x$geom, "GeomText"), s6_b$layers)
w1_strip <- setNames(paste0(label_cohort(wq$cohort), "\nW1 = ", sprintf("%.2f", wq$W1_hours), " h"),
    wq$cohort)
s6_b <- s6_b + facet_wrap(~cohort, ncol = 1, labeller = labeller(cohort = w1_strip)) + polar_grid_theme
# S6 scales are defined in Cell 10; do not override the cohort-specific display groups here.
s6_c <- s6_c + labs(x = "log2(2 x amplitude + 1)")
# Figure 2K uses contrast-aware text colours from its BH-q scale in Cell 11.
s5_j$data$text_colour <- ifelse(s5_j$data$fold_enrichment > 2.4, "white", "black")
s5_j$layers[[2]]$mapping <- aes(label = label_fe, colour = text_colour)
s5_j <- s5_j + scale_colour_identity()
s5_k$data$text_colour <- ifelse(s5_k$data$neg_log10_p > 4, "white", "black")
s5_k$layers[[2]]$mapping <- aes(label = label_logp, colour = text_colour)
s5_k <- s5_k + scale_colour_identity()
# Plain-text mathematical labels keep every glyph at the configured role size.
fig2_j <- fig2_j + labs(title = expression("Axial polarization " * R[textstyle(2)]),
    y = expression("Axial " * R[textstyle(2)]))
s5_g <- s5_g + labs(x = "Delta R2")
s6_a <- s6_a + labs(y = expression("-log10(" * italic(p) * ")"))
chip_fill_scale <- fig2_k$scales$get_scales("fill")
chip_fill_scale$name <- expression("-log10(" * italic(q) * ")")
ppi_fill_scale <- s5_k$scales$get_scales("fill")
ppi_fill_scale$name <- expression("-log10(" * italic(p) * ")")

# Compact K/L typography keeps the global font roles at >=7 pt.
fig2_k <- fig2_k + labs(subtitle = NULL) +
    scale_x_discrete(labels = c(clock_tf_labels, ANY_CCG_PEAK = "Pooled\nCCG"), expand = expansion(add = 0)) +
    scale_y_discrete(labels = c(GTEx_Normal = "GTEx", `TCGA-PAAD_Tumor` = "TCGA-\nspecific"), expand = expansion(add = 0)) +
    theme(axis.text.x = element_text(size = FONT$tick, angle = 45, hjust = 1,
            margin = margin(t = 1)),
        axis.text.y = element_text(size = FONT$tick, margin = margin(r = 2)),
        plot.title = element_text(size = FONT$title, face = "bold", margin = margin(b = 2)),
        plot.margin = margin(0.6, 1.8, 0.3, 0.6, "mm"),
        legend.margin = margin(0, 0, 0, 0), legend.box.spacing = grid::unit(0, "mm"),
        legend.title = element_text(size = FONT$legend), legend.text = element_text(size = FONT$legend)) +
    theme(legend.position = "right", legend.direction = "vertical") +
    guides(fill = guide_colourbar(direction = "vertical", title.position = "top",
        barwidth = grid::unit(2, "mm"), barheight = grid::unit(14, "mm")))
fig2_l <- fig2_l + scale_y_discrete(labels = c(GTEx_Normal = "GTEx", `TCGA-PAAD_Tumor` = "TCGA-specific")) +
    theme(plot.title = element_text(size = FONT$title, face = "bold", margin = margin(b = 1)),
        axis.title.x = element_text(size = FONT$axis_title, margin = margin(t = 1)),
        axis.text = element_text(size = FONT$tick),
        axis.text.x = element_text(size = FONT$tick, margin = margin(t = 1)),
        plot.margin = margin(0.6, 3, 0.3, 1.2, "mm"), axis.ticks.length = grid::unit(0.8, "mm"))

# Requested panel headings and explicit bounds; all numerical results are unchanged.
fig2_h <- fig2_h + labs(title = "Rhythmic gene overlap\nTCGA vs GTEx") +
    scale_y_continuous(limits = c(0, 15000), breaks = seq(0, 15000, 5000), expand = c(0, 0))
fig2_i$layers[[1]]$aes_params$linewidth <- 0.8
fig2_i <- fig2_i + labs(title = "Acrophase distribution", x = "Circadian time (h)")
fig2_l <- fig2_l + scale_x_continuous(limits = c(0, 1.5), breaks = seq(0, 1.5, 0.5), expand = c(0, 0))

# S5 display-only alignment overrides for Cell 23.
# Apply after the existing compact panel styling block and before Cell 13 exports.

# S5 polar panels use the same enlarged, point-adjacent label treatment as Figure 2B/D.
# Unlike the Figure 2 gene-label exception, all S5 labels remain at the 7-pt annotation size.
s5_direct_polar <- function(plot, phase_col, radius_col, title, radial_title,
                            point_colour = NULL, fill_col = NULL) {
    point_layer <- which(vapply(plot$layers, function(layer) inherits(layer$geom, "GeomPoint"), logical(1)))
    stopifnot(length(point_layer) == 1L)
    d <- plot$layers[[point_layer]]$data
    out <- clock_polar_frame(d, phase_col, radius_col, title, radial_title,
        point_colour = point_colour, fill_col = fill_col, direct_labels = TRUE)
    label_layer <- which(vapply(out$layers, function(layer) inherits(layer$geom, "GeomTextRepel"), logical(1)))
    stopifnot(length(label_layer) == 1L)
    out$layers[[label_layer]]$aes_params$size <- FONT$annotation / ggplot2::.pt
    out$layers[[label_layer]]$geom_params$force <- 3
    out$layers[[label_layer]]$geom_params$force_pull <- 0.1
    out$layers[[label_layer]]$geom_params$box.padding <- grid::unit(0.18, "lines")
    out$layers[[label_layer]]$geom_params$max.iter <- 100000
    out$layers[[label_layer]]$geom_params$max.time <- Inf
    radial_layer <- which(vapply(out$layers, function(layer)
        inherits(layer$geom, "GeomText") && !"hour" %in% names(layer$data), logical(1)))
    radial_data <- out$layers[[radial_layer]]$data
    out$layers[[label_layer]]$data <- bind_rows(out$layers[[label_layer]]$data,
        transmute(radial_data, x = x - 0.09, y = y, label = ""))
    out$layers[[label_layer]]$geom_params$point.padding <- grid::unit(0.4, "lines")

    out + labs(y = radial_title) +
        theme(plot.title = element_text(hjust = 0.5),
            plot.margin = margin(0.8, 0.8, 0.6, 0.6, unit = "mm"))
}

s5_a <- s5_direct_polar(s5_a, "phase_h", "loading", "GTEx", "Mode 1 loading (100%)",
    point_colour = COHORT_COLOURS[["GTEx_Normal"]])
s5_b <- s5_direct_polar(s5_b, "phase_h", "loading", "TCGA", "Mode 1 loading (100%)",
    point_colour = COHORT_COLOURS[["TCGA-PAAD_Tumor"]])
s5_c <- s5_direct_polar(s5_c, "phase_h", "loading", "CPTAC", "Mode 1 loading (100%)",
    point_colour = COHORT_COLOURS[["CPTAC-3_Tumor"]])
s5_d <- s5_direct_polar(s5_d, "acrophase", "Amplitude", "CPTAC", "Amplitude",
    fill_col = "neglog10p")
s5_h <- s5_direct_polar(s5_h, "phase_h", "loading", "GTEx ToD [0, 12)", "Mode 1 loading (100%)",
    point_colour = COHORT_COLOURS[["GTEx_Normal"]])

# Keep the CPTAC overlap panel on the Figure 2H bar scale and title treatment.
s5_e <- overlap_bar("CPTAC-3_Tumor", bar_width = 0.45) +
    labs(title = "Rhythmic gene overlap\nCPTAC vs GTEx") +
    scale_y_continuous(limits = c(0, 15000), breaks = seq(0, 15000, 5000), expand = c(0, 0)) +
    theme(plot.title = element_text(hjust = 0.5))

# Use a true R subscript and retain the comparison in the title.
s5_g <- s5_g + labs(title = "TCGA minus GTEx",
    x = expression(Delta * " axial " * R[textstyle(2)]), y = "Permutations")

# Match the Figure 2 density panel's title, time axis, and trace weight.
s5_i$layers[[2]]$aes_params$linewidth <- 0.8
s5_i <- s5_i + labs(title = "GTEx ToD [0, 12): gene phases",
    x = "Circadian time (h)", y = "Density")

# Spell out the cancer-specific tumor target in both per-clock PPI heatmaps.
s5_tumor_rows <- c(GTEx_Normal = "GTEx", `TCGA-PAAD_Tumor` = "TCGA-specific",
    `CPTAC-3_Tumor` = "CPTAC-specific")
s5_j <- s5_j + scale_y_discrete(labels = s5_tumor_rows) +
    theme(axis.text.y = element_text(size = FONT$tick, margin = margin(r = 2)))
s5_k <- s5_k + scale_y_discrete(labels = s5_tumor_rows) +
    theme(axis.text.y = element_text(size = FONT$tick, margin = margin(r = 2)))


In [ ]:
# Cell 13: Individual panels and assembled exports
# Add transparent panel-letter text on the full panel canvas. This
# preserves centered titles and fixed-aspect panels without extra layout rows.
pdf_tag <- function(p, letter) {
    p <- p + theme(plot.title = element_text(hjust = 0.5), plot.title.position = "plot")
    plot_grob <- if (inherits(p, "patchwork")) patchwork::patchworkGrob(p) else ggplot2::ggplotGrob(p)
    tag_grob <- grid::textGrob(letter, x = grid::unit(1, "mm"),
        y = grid::unit(1, "npc") - grid::unit(1, "mm"), just = c("left", "top"),
        gp = grid::gpar(fontfamily = FONT_FAMILY, fontsize = FONT$tag, fontface = "bold"))
    grid::grobTree(plot_grob, tag_grob)
}

row_of <- function(plots, widths = rep(1, length(plots))) {
    wrap_plots(lapply(plots, function(p) wrap_elements(full = p)), nrow = 1, widths = widths)
}
assemble_fig2 <- function(p) {
    top <- row_of(p[c("A", "B")], c(3, 1))
    second <- row_of(p[c("C", "D")], c(3, 1))
    third <- row_of(p[c("E", "F", "G", "H")])
    lower_right <- (wrap_elements(full = p$K)/wrap_elements(full = p$L)) + plot_layout(heights = c(0.6,
        0.4))
    bottom <- row_of(list(p$I, p$J, lower_right), c(1, 1, 2))
    (wrap_elements(full = top)/wrap_elements(full = second)/wrap_elements(full = third)/wrap_elements(full = bottom)) +
        plot_layout(heights = c(1, 1, 2, 1))
}
fig2_j <- fig2_j + labs(title = expression(bold("Axial polarization " * R[textstyle(2)])))
fig2_panels <- list(A = fig2_a, B = fig2_b, C = fig2_c, D = fig2_d, E = fig2_e, F = fig2_f, G = fig2_g,
    H = fig2_h, I = fig2_i, J = fig2_j, K = fig2_k, L = fig2_l)
fig2_sizes <- list(A = c(3, 1), B = c(1, 1), C = c(3, 1), D = c(1, 1), E = c(1, 2), F = c(1, 2), G = c(1,
    2), H = c(1, 2), I = c(1, 1), J = c(1, 1), K = c(2, 0.6), L = c(2, 0.4))
fig2_content <- c(A = "gtex_clock_expression", B = "gtex_clock_amplitude_phase", C = "tcga_clock_expression",
    D = "tcga_clock_amplitude_phase", E = "gtex_rhythmic_heatmap", F = "tcga_rhythmic_heatmap", G = "tcga_heatmap_gtex_acrophase_order",
    H = "gtex_tcga_rhythmic_overlap", I = "acrophase_distribution", J = "axial_polarization", K = "clock_tf_chip_enrichment",
    L = "ccg_ppi_enrichment")
fig2_tagged <- setNames(lapply(names(fig2_panels), function(x) pdf_tag(fig2_panels[[x]], x)), names(fig2_panels))
for (id in names(fig2_panels)) save_publication(fig2_panels[[id]], paste0("figure2/Figure2", id, "_",
    fig2_content[[id]]), fig2_sizes[[id]][1], fig2_sizes[[id]][2], fig2_tagged[[id]])
save_publication(assemble_fig2(fig2_panels), "figure2/Figure2_assembled_cohort_clock_organization", 4,
    5, assemble_fig2(fig2_tagged))

s4_panels <- list(A = s4_a, B = s4_b)
s4_tagged <- list(A = pdf_tag(s4_a, "A"), B = pdf_tag(s4_b, "B"))
save_publication(s4_a, "supplementary4/FigureS4A_tcga_core_clock_expression", 4, 3, s4_tagged$A)
save_publication(s4_b, "supplementary4/FigureS4B_gtex_core_clock_expression", 4, 3, s4_tagged$B)
assemble_s4 <- function(p) (wrap_elements(full = p$A)/wrap_elements(full = p$B)) + plot_layout(heights = c(1,
    1))
save_publication(assemble_s4(s4_panels), "supplementary4/FigureS4_assembled_core_clock_expression", 4,
    6, assemble_s4(s4_tagged))

s5_panels <- list(A = s5_a, B = s5_b, C = s5_c, D = s5_d, E = s5_e, F = s5_f, G = s5_g, H = s5_h, I = s5_i,
    J = s5_j, K = s5_k)
s5_sizes <- list(A = c(1, 1), B = c(1, 1), C = c(1, 1), D = c(1, 1), E = c(1, 1), F = c(2, 1), G = c(1,
    1), H = c(2, 1), I = c(2, 1), J = c(2, 1), K = c(2, 1))
s5_content <- c(A = "gtex_core_clock_svd", B = "tcga_core_clock_svd", C = "cptac_core_clock_svd", D = "cptac_clock_amplitude_phase",
    E = "gtex_cptac_rhythmic_overlap", F = "three_cohort_rhythmic_venn", G = "axial_polarization_permutation",
    H = "gtex_tod_clock_svd", I = "gtex_tod_acrophase_distribution", J = "clock_gene_ppi_fold_enrichment",
    K = "clock_gene_ppi_significance")
s5_tagged <- setNames(lapply(names(s5_panels), function(x) pdf_tag(s5_panels[[x]], x)), names(s5_panels))
for (id in names(s5_panels)) save_publication(s5_panels[[id]], paste0("supplementary5/FigureS5", id,
    "_", s5_content[[id]]), s5_sizes[[id]][1], s5_sizes[[id]][2], s5_tagged[[id]])
assemble_s5 <- function(p) {
    rows <- list(row_of(p[c("A", "B", "C", "D")]), row_of(p[c("E", "F", "G")], c(1, 2, 1)), row_of(p[c("H",
        "I")], c(2, 2)), row_of(p[c("J", "K")], c(2, 2)))
    wrap_plots(lapply(rows, function(p) wrap_elements(full = p)), ncol = 1, heights = c(1, 1, 1, 1))
}
save_publication(assemble_s5(s5_panels), "supplementary5/FigureS5_assembled_cohort_sensitivity_and_enrichment",
    4, 4, assemble_s5(s5_tagged))

s6_panels <- list(A = s6_a, B = s6_b, C = s6_c)
s6_tagged <- setNames(lapply(names(s6_panels), function(x) pdf_tag(s6_panels[[x]], x)), names(s6_panels))
s6_content <- c(A = "common_unique_pvalue_distribution", B = "common_unique_acrophase_distribution",
    C = "common_unique_amplitude_accumulation")
for (id in names(s6_panels)) save_publication(s6_panels[[id]], paste0("supplementary6/FigureS6", id,
    "_", s6_content[[id]]), if (id == "C") 2 else 1, 2, s6_tagged[[id]])
save_publication(row_of(s6_panels, c(1, 1, 2)), "supplementary6/FigureS6_assembled_common_unique_rhythmic_genes",
    4, 2, row_of(s6_tagged, c(1, 1, 2)))



## 7. Reproducibility record and export checks

In [ ]:
# Cell 14: Validation, provenance, and software versions
export_manifest <- bind_rows(EXPORT_MANIFEST)
allowed_minimum <- ifelse(export_manifest$stem %in% c("figure2/Figure2B_gtex_clock_amplitude_phase",
    "figure2/Figure2D_tcga_clock_amplitude_phase", "figure2/Figure2_assembled_cohort_clock_organization"),
    FONT$gene_annotation, 7)
stopifnot(nrow(export_manifest) == 32L, !anyDuplicated(export_manifest$stem),
    all(export_manifest$minimum_text_pt >= allowed_minimum - 1e-06))
files <- file.path(FIGURE_DIR, c(export_manifest$png, export_manifest$pdf))
stopifnot(all(file.exists(files)), all(file.info(files)$size > 0))
write_csv(export_manifest, file.path(TABLE_DIR, "publication_export_manifest.csv"))
write_csv(tibble(role = names(FONT), font_size_pt = unlist(FONT), family = FONT_FAMILY), file.path(TABLE_DIR,
    "publication_font_configuration.csv"))
manifest_inputs <- unique(c(unname(input_files),
    file.path(MERGED_DIR, "regression", paste0("sig_genes_regression_periodic_12h_", COHORTS, ".txt")),
    file.path(DATA_DIR, "Ref", "genes.bed"),
    file.path(DATA_DIR, "gtex_chiral_sensitivity_supplement",
        c("harmonic_regression_all_genes.tsv", "rhythmic_genes.tsv", "chiral_phase_estimates.tsv")),
    unname(chip_input_paths), unname(current_regression_paths), ppi_background_path, ppi_partners_path))
stopifnot(all(file.exists(manifest_inputs)))
input_manifest <- tibble(path_relative_to_DATA_DIR = substring(manifest_inputs, nchar(DATA_DIR) + 2),
    bytes = file.info(manifest_inputs)$size, sha256 = vapply(manifest_inputs, digest::digest, character(1),
        algo = "sha256", file = TRUE))
write_csv(input_manifest, file.path(TABLE_DIR, "publication_input_manifest.csv"))
writeLines(capture.output(sessionInfo()), file.path(TABLE_DIR, "publication_session_info.txt"))
print(export_manifest |>
    count(figure = dirname(stem), name = "png_pdf_pairs"))
cat("Validated 32 PNG/PDF pairs; text is at least 7 pt except the requested 5-pt B/D gene labels.\n")
sessionInfo()

